In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/__huggingface_repos__.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/validation_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/split_manifest.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/train_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/best_strict_macro_f1.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/validation_error_analysis.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/.cache/huggingface/.gitignore
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/.cache/huggingface/CACHEDIR.TAG
/kaggle/input/

In [2]:
"""
Oracle round-trip test: gold spans -> token BIO labels -> decoded char spans -> strict F1 vs gold.
Must give ~1.0. If not, fix preprocessing/decoding BEFORE training anything.

Edit the paths below for your Kaggle environment.
"""
import json
from collections import defaultdict
from tokenizers import Tokenizer

ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER/Train"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"

tok = Tokenizer.from_file(TOK)
labels = [l.strip() for l in open(f"{ROOT}/labels.txt") if l.strip()]
print("labels:", labels)

text = json.load(open(f"{ROOT}/fold1-text.json"))
gold = json.load(open(f"{ROOT}/fold1-gold.json"))

# Inspect gold format and check that text[start:end] matches the stored entity text
bad_slice = 0
total = 0
for d, ents in gold.items():
    for e in ents:
        total += 1
        if len(e) > 3 and text[d][e[0]:e[1]] != e[3]:
            bad_slice += 1
print(f"gold spans: {total}, text[start:end] != entity_text: {bad_slice}")


def encode_bio(doc, ents):
    enc = tok.encode(doc, add_special_tokens=False)
    offs = enc.offsets
    tags = ["O"] * len(offs)
    for s, e, lab, *_ in ents:
        idx = [i for i, (a, b) in enumerate(offs) if a < e and b > s]  # token overlaps span
        for k, i in enumerate(idx):
            tags[i] = ("B-" if k == 0 else "I-") + lab
    return offs, tags


def decode_bio(doc, offs, tags):
    out, cur = [], None
    for (a, b), t in zip(offs, tags):
        if t == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = t.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: out.append(cur)
    # trim whitespace so boundaries are tight
    res = []
    for s, e, lab in out:
        while s < e and doc[s].isspace(): s += 1
        while e > s and doc[e - 1].isspace(): e -= 1
        res.append((s, e, lab))
    return res


tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
for d, ents in gold.items():
    offs, tags = encode_bio(text[d], ents)
    pred = set(decode_bio(text[d], offs, tags))
    ref = {(e[0], e[1], e[2]) for e in ents}
    for x in pred & ref: tp[x[2]] += 1
    for x in pred - ref: fp[x[2]] += 1
    for x in ref - pred: fn[x[2]] += 1

f1s = []
for l in labels:
    p = tp[l] / max(tp[l] + fp[l], 1); r = tp[l] / max(tp[l] + fn[l], 1)
    f = 2 * p * r / max(p + r, 1e-9); f1s.append(f)
    print(f"{l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}  (missed={fn[l]}, extra={fp[l]})")
print("ORACLE strict macro-F1 ceiling for this tokenizer:", sum(f1s) / len(f1s))

labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
gold spans: 4120, text[start:end] != entity_text: 0
APP        P=0.976 R=0.976 F1=0.976  (missed=5, extra=5)
RESP       P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
A.COUNSEL  P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
R.COUNSEL  P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
JUDGE      P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
WIT        P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
AUTH       P=0.981 R=0.981 F1=0.981  (missed=10, extra=10)
COURT      P=0.987 R=0.987 F1=0.987  (missed=6, extra=6)
STAT       P=0.989 R=0.989 F1=0.989  (missed=6, extra=6)
PREC       P=0.936 R=0.936 F1=0.936  (missed=40, extra=40)
DATE       P=1.000 R=1.000 F1=1.000  (missed=0, extra=0)
CASENO     P=0.998 R=0.998 F1=0.998  (missed=1, extra=1)
ORACLE strict macro-F1 ceiling for this tokenizer: 0.9888393172442642


In [4]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    raise NotImplementedError("Provide the Phase-1 model class + checkpoint loading")


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split()
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch])
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{a.arch}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{a.arch}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{a.arch}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{a.arch}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{a.arch}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{a.arch}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

train docs 62 val docs 8 windows 1062


NotImplementedError: Provide the Phase-1 model class + checkpoint loading

In [6]:
import torch
p = "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
ck = torch.load(p, map_location="cpu", weights_only=False)
print(type(ck))
if isinstance(ck, dict):
    print("top-level keys:", list(ck.keys())[:15])
sd = ck.get("model", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck.state_dict()
print("num tensors:", len(sd))
for k in list(sd.keys())[:25]:
    print(k, tuple(sd[k].shape))
print("...")
for k in list(sd.keys())[-8:]:
    print(k, tuple(sd[k].shape))
if isinstance(ck, dict):
    for k in ("config", "cfg", "args", "model_args"):
        if k in ck: print(k, "=", ck[k])

<class 'dict'>
top-level keys: ['step', 'model', 'best_val', 'cfg']
num tensors: 100
token_embedding.weight (40000, 1024)
layers.0.norm1.weight (1024,)
layers.0.norm1.bias (1024,)
layers.0.attn.qkv_proj.weight (2048, 1024)
layers.0.attn.out_proj.weight (1024, 1024)
layers.0.norm2.weight (1024,)
layers.0.norm2.bias (1024,)
layers.0.ffn.in_proj.weight (6656, 1024)
layers.0.ffn.out_proj.weight (1024, 3328)
layers.1.norm1.weight (1024,)
layers.1.norm1.bias (1024,)
layers.1.attn.qkv_proj.weight (2048, 1024)
layers.1.attn.out_proj.weight (1024, 1024)
layers.1.norm2.weight (1024,)
layers.1.norm2.bias (1024,)
layers.1.ffn.in_proj.weight (6656, 1024)
layers.1.ffn.out_proj.weight (1024, 3328)
layers.2.norm1.weight (1024,)
layers.2.norm1.bias (1024,)
layers.2.attn.qkv_proj.weight (2048, 1024)
layers.2.attn.out_proj.weight (1024, 1024)
layers.2.norm2.weight (1024,)
layers.2.norm2.bias (1024,)
layers.2.ffn.in_proj.weight (6656, 1024)
layers.2.ffn.out_proj.weight (1024, 3328)
...
layers.11.attn.out_

In [7]:
"""
Arch4 (hybrid sliding-window/global GQA, 201M) reconstruction from checkpoint shapes.

STEP 1 (once):   run_check()   -> tries 16 layout variants on Phase-1 val tokens, saves the best to VARIANT_PATH
STEP 2:          build_arch4(path) -> (bidirectional hidden-state backbone, 1024) for the NER pipeline
"""
import itertools, json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch4_variant.json"
DEFAULT_V = {"gl": 0, "rope": 0, "swi": 0, "qkv": "cat"}


def rope_cache(T, hd, base, device):
    inv = 1.0 / (base ** (torch.arange(0, hd, 2, device=device).float() / hd))
    fr = torch.outer(torch.arange(T, device=device).float(), inv)
    return fr.cos(), fr.sin()


def apply_rope(x, cos, sin, interleaved):  # x [B,H,T,hd]
    cos, sin = cos[None, None], sin[None, None]
    if interleaved:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1)


class Attn(nn.Module):
    def __init__(self, d, nq, nkv):
        super().__init__()
        self.nq, self.nkv, self.hd = nq, nkv, d // nq
        self.qkv_proj = nn.Linear(d, (nq + 2 * nkv) * self.hd, bias=False)
        self.out_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin, mask, V):
        B, T, _ = x.shape
        nq, nkv, hd = self.nq, self.nkv, self.hd
        rep = nq // nkv
        qkv = self.qkv_proj(x)
        if V["qkv"] == "cat":
            q, k, v = qkv.split([nq * hd, nkv * hd, nkv * hd], -1)
            q, k, v = q.view(B, T, nq, hd), k.view(B, T, nkv, hd), v.view(B, T, nkv, hd)
        else:  # grouped per kv head: [q*rep, k, v]
            g = qkv.view(B, T, nkv, rep + 2, hd)
            q = g[:, :, :, :rep].reshape(B, T, nq, hd); k = g[:, :, :, rep]; v = g[:, :, :, rep + 1]
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        q = apply_rope(q, cos, sin, V["rope"]); k = apply_rope(k, cos, sin, V["rope"])
        k = k.repeat_interleave(rep, 1); v = v.repeat_interleave(rep, 1)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, -1))


class FFN(nn.Module):
    def __init__(self, d, dff):
        super().__init__()
        self.in_proj = nn.Linear(d, 2 * dff, bias=False)
        self.out_proj = nn.Linear(dff, d, bias=False)

    def forward(self, x, V):
        a, b = self.in_proj(x).chunk(2, -1)
        return self.out_proj(F.silu(a) * b if V["swi"] == 0 else F.silu(b) * a)


class Block(nn.Module):
    def __init__(self, d, nq, nkv, dff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d); self.attn = Attn(d, nq, nkv)
        self.norm2 = nn.LayerNorm(d); self.ffn = FFN(d, dff)

    def forward(self, x, cos, sin, mask, V):
        x = x + self.attn(self.norm1(x), cos, sin, mask, V)
        return x + self.ffn(self.norm2(x), V)


class Arch4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg["D_MODEL"]
        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], d)
        self.layers = nn.ModuleList([Block(d, cfg["N_Q_HEADS"], cfg["N_KV_HEADS"], cfg["D_FF"])
                                     for _ in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["VOCAB_SIZE"], bias=False)
        self.V = dict(DEFAULT_V)

    def is_global(self, i):
        ge = self.cfg["GLOBAL_EVERY"]
        return (i + 1) % ge == 0 if self.V["gl"] == 0 else i % ge == 0

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev = ids.device
        win = self.cfg["WINDOW"]
        i = torch.arange(T, device=dev)[:, None]; j = torch.arange(T, device=dev)[None]
        dist = i - j
        if causal: loc, glob = (dist >= 0) & (dist < win), dist >= 0
        else: loc, glob = dist.abs() < win, torch.ones(T, T, dtype=torch.bool, device=dev)
        loc, glob = loc[None, None], glob[None, None]
        if pad_mask is not None:
            key = pad_mask.bool()[:, None, None, :]
            loc, glob = loc & key, glob & key
        cos, sin = rope_cache(T, self.cfg["D_MODEL"] // self.cfg["N_Q_HEADS"], self.cfg["ROPE_BASE"], dev)
        x = self.token_embedding(ids)
        for n, blk in enumerate(self.layers):
            x = blk(x, cos, sin, glob if self.is_global(n) else loc, self.V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    """Bidirectional hidden-state view of the (causally pretrained) model."""
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = Arch4(ck["cfg"]); m.load_state_dict(ck["model"], strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=4, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best_val:", ck.get("best_val"), "| step:", ck.get("step"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= m.cfg["VOCAB_SIZE"]: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    batches = [b for b in batches if len(b) == seq_len + 1]
    x = torch.stack(batches).to(dev)
    res = []
    for gl, rope, swi, qkv in itertools.product([0, 1], [0, 1], [0, 1], ["cat", "grouped"]):
        m.V = {"gl": gl, "rope": rope, "swi": swi, "qkv": qkv}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V))); print(f"{loss:8.4f}  {m.V}")
    res.sort(key=lambda r: r[0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0])
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH, "(loss should be close to best_val; clearly lower than all others)")


def build_arch4(path=CKPT, causal=False):
    dev = "cpu"
    m, _ = load_model(path, dev)
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    m.lm_head = nn.Identity()  # not needed for NER; saves memory (state already loaded)
    return NERBackbone(m, causal=causal), m.cfg["D_MODEL"]


if __name__ == "__main__":
    run_check()

checkpoint best_val: 2.475979009747505 | step: 10000
  2.2604  {'gl': 0, 'rope': 0, 'swi': 0, 'qkv': 'cat'}
  7.4957  {'gl': 0, 'rope': 0, 'swi': 0, 'qkv': 'grouped'}
  4.7490  {'gl': 0, 'rope': 0, 'swi': 1, 'qkv': 'cat'}
  8.2485  {'gl': 0, 'rope': 0, 'swi': 1, 'qkv': 'grouped'}
  4.2593  {'gl': 0, 'rope': 1, 'swi': 0, 'qkv': 'cat'}
  7.6124  {'gl': 0, 'rope': 1, 'swi': 0, 'qkv': 'grouped'}
  6.2932  {'gl': 0, 'rope': 1, 'swi': 1, 'qkv': 'cat'}
  8.3156  {'gl': 0, 'rope': 1, 'swi': 1, 'qkv': 'grouped'}
  2.6234  {'gl': 1, 'rope': 0, 'swi': 0, 'qkv': 'cat'}
  7.4198  {'gl': 1, 'rope': 0, 'swi': 0, 'qkv': 'grouped'}
  5.3667  {'gl': 1, 'rope': 0, 'swi': 1, 'qkv': 'cat'}
  8.1301  {'gl': 1, 'rope': 0, 'swi': 1, 'qkv': 'grouped'}
  4.3116  {'gl': 1, 'rope': 1, 'swi': 0, 'qkv': 'cat'}
  7.5284  {'gl': 1, 'rope': 1, 'swi': 0, 'qkv': 'grouped'}
  6.3226  {'gl': 1, 'rope': 1, 'swi': 1, 'qkv': 'cat'}
  8.2477  {'gl': 1, 'rope': 1, 'swi': 1, 'qkv': 'grouped'}

BEST: (np.float64(2.26035137971242

In [8]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    if arch != "arch4":
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    try:
        fn = importlib.import_module("arch4_backbone").build_arch4
    except ImportError:  # notebook: run the arch4_backbone cell first so build_arch4 is a global
        fn = globals()["build_arch4"]
    return fn(ckpt_path)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split()
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch])
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{a.arch}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{a.arch}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{a.arch}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{a.arch}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{a.arch}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{a.arch}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

train docs 62 val docs 8 windows 1062


/tmp/ipykernel_48/1969851411.py:250: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_48/1969851411.py:262: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.5323 val strict macroF1 0.3512
   APP        P=0.071 R=0.016 F1=0.026
   RESP       P=0.200 R=0.167 F1=0.182
   A.COUNSEL  P=0.000 R=0.000 F1=0.000
   R.COUNSEL  P=0.036 R=0.250 F1=0.063
   JUDGE      P=0.487 R=0.655 F1=0.559
   WIT        P=0.000 R=0.000 F1=0.000
   AUTH       P=0.185 R=0.220 F1=0.201
   COURT      P=0.733 R=0.907 F1=0.811
   STAT       P=0.594 R=0.829 F1=0.693
   PREC       P=0.403 R=0.484 F1=0.440
   DATE       P=0.841 R=0.964 F1=0.898
   CASENO     P=0.248 R=0.549 F1=0.342
ep1 loss 0.1081 val strict macroF1 0.3921
   APP        P=0.294 R=0.159 F1=0.206
   RESP       P=0.120 R=0.100 F1=0.109
   A.COUNSEL  P=1.000 R=0.071 F1=0.133
   R.COUNSEL  P=0.103 R=0.333 F1=0.157
   JUDGE      P=0.281 R=0.310 F1=0.295
   WIT        P=0.000 R=0.000 F1=0.000
   AUTH       P=0.396 R=0.331 F1=0.361
   COURT      P=0.771 R=0.938 F1=0.847
   STAT       P=0.660 R=0.736 F1=0.696
   PREC       P=0.273 R=0.469 F1=0.345
   DATE       P=0.935 R=0.976 F1=0.955
   CASENO     P=0.5

RuntimeError: cannot reshape tensor of 0 elements into shape [1, 0, -1] because the unspecified dimension size -1 can be any value and is ambiguous

In [10]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    if arch != "arch4":
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    try:
        fn = importlib.import_module("arch4_backbone").build_arch4
    except ImportError:  # notebook: run the arch4_backbone cell first so build_arch4 is a global
        fn = globals()["build_arch4"]
    return fn(ckpt_path)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split()
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch])
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{a.arch}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{a.arch}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{a.arch}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{a.arch}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{a.arch}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{a.arch}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

train docs 62 val docs 8 windows 1062


/tmp/ipykernel_48/973665838.py:257: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")


wrote /kaggle/working/submission_arch4.zip | best val: -1


In [11]:
import zipfile, json
z = zipfile.ZipFile("/kaggle/working/submission_arch4.zip")
print(z.namelist())                       # must be exactly ['prediction.json']
p = json.loads(z.read("prediction.json"))
test = json.load(open("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER/Test/fold3-text.json"))
print(len(p), "docs predicted vs", len(test), "test docs")
print(sum(len(v) for v in p.values()), "total entities")
d = next(k for k in p if p[k])
for s, e, l, t in p[d][:5]:
    assert test[d][s:e] == t
    print(l, repr(t))

['prediction.json']
35 docs predicted vs 35 test docs
4424 total entities
COURT 'SUPREME COURT OF INDIA'
CASENO 'CIVIL APPEAL NO.6580 OF 2008'
APP 'ARISING'
CASENO 'SPECIAL LEAVE PETITION (CIVIL) NO. 15449 OF'
APP 'ESSCO FABS PVT. LTD.'


In [12]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    if arch != "arch4":
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    try:
        fn = importlib.import_module("arch4_backbone").build_arch4
    except ImportError:  # notebook: run the arch4_backbone cell first so build_arch4 is a global
        fn = globals()["build_arch4"]
    return fn(ckpt_path)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    tag = a.arch + (f"_f{a.fold}" if a.fold >= 0 else "")
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch])
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

WIT training tokens: 577 | role_boost 1.0
train docs 62 val docs 8 windows 1062


/tmp/ipykernel_48/2193221717.py:280: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")


wrote /kaggle/working/submission_arch4.zip | best val: -1


In [13]:
import sys, gc, torch
for f in range(5):
    sys.argv = ["x", "--arch", "arch4", "--fold", str(f), "--epochs", "6",
                "--lr_bb", "2e-5", "--role_boost", "2.0"]
    main()
    gc.collect(); torch.cuda.empty_cache()

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891


/tmp/ipykernel_48/2193221717.py:280: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_48/2193221717.py:292: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.5912 val strict macroF1 0.3067
   APP        P=0.130 R=0.145 F1=0.137
   RESP       P=0.010 R=0.012 F1=0.011
   A.COUNSEL  P=0.125 R=0.135 F1=0.130
   R.COUNSEL  P=0.088 R=0.286 F1=0.135
   JUDGE      P=0.300 R=0.400 F1=0.343
   WIT        P=0.049 R=0.105 F1=0.067
   AUTH       P=0.446 R=0.599 F1=0.512
   COURT      P=0.337 R=0.605 F1=0.433
   STAT       P=0.479 R=0.622 F1=0.541
   PREC       P=0.107 R=0.147 F1=0.124
   DATE       P=0.740 R=0.894 F1=0.810
   CASENO     P=0.349 R=0.593 F1=0.439
ep1 loss 0.1338 val strict macroF1 0.3781
   APP        P=0.219 R=0.193 F1=0.205
   RESP       P=0.100 R=0.082 F1=0.090
   A.COUNSEL  P=0.312 R=0.676 F1=0.427
   R.COUNSEL  P=0.364 R=0.095 F1=0.151
   JUDGE      P=0.397 R=0.450 F1=0.422
   WIT        P=0.056 R=0.105 F1=0.073
   AUTH       P=0.258 R=0.281 F1=0.269
   COURT      P=0.617 R=0.768 F1=0.684
   STAT       P=0.615 R=0.655 F1=0.634
   PREC       P=0.218 R=0.234 F1=0.226
   DATE       P=0.890 R=0.908 F1=0.899
   CASENO     P=0.3

In [14]:
import sys
sys.argv = ["x", "--arch", "arch4", "--ensemble"]
main()      # writes /kaggle/working/submission_arch4_ens.zip

models averaged: 5 | docs: 35 | entities: 4155


In [15]:
import torch
p = "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt"
ck = torch.load(p, map_location="cpu", weights_only=False)
print(type(ck))
if isinstance(ck, dict):
    print("top-level keys:", list(ck.keys())[:15])
    for k in ("cfg", "config", "args", "model_args", "best_val", "step"):
        if k in ck: print(k, "=", ck[k])
sd = ck.get("model", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck.state_dict()
print("num tensors:", len(sd))
for k in list(sd.keys())[:30]:
    print(k, tuple(sd[k].shape))
print("...")
for k in list(sd.keys())[-8:]:
    print(k, tuple(sd[k].shape))

<class 'dict'>
top-level keys: ['model', 'opt', 'scaler', 'step', 'best', 'cfg', 'n_params']
cfg = {'BLOCK': 1024, 'MICRO_BATCH': 1, 'GRAD_ACCUM': 16, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}
step = 10000
num tensors: 141
token_emb.weight (40000, 768)
pos_emb.weight (1024, 768)
blocks.0.norm1.weight (768,)
blocks.0.norm1.bias (768,)
blocks.0.attn.mask (1, 1, 1024, 1024)
blocks.0.attn.q_proj.weight (768, 768)
blocks.0.attn.q_proj.bias (768,)
blocks.0.attn.k_proj.weight (768, 768)
blocks.0.attn.k_proj.bias (768,)
blocks.0.attn.v_proj.weight (768, 768)
blocks.0.attn.v_proj.bias (768,)
blocks.0.attn.out_proj.weight (768, 768)
blocks.0.attn.out_proj.bias (768,)
blocks.0.norm2.weight (768,)
blocks.0.norm2.bias (768,)
blocks.0.mlp.fc1.weight (3072, 768)
blocks.0.mlp.fc1.bias (3072,)
blocks.0.mlp.fc2.weight (768, 3072)
blocks.0.mlp.fc2.bias (768,)
blocks.1.norm1.wei

In [16]:
"""
Arch1 (GPT-2 style, 8 layers, d=768, learned pos-emb, tied lm_head, 88M) reconstruction from checkpoint shapes.

STEP 1 (once):  run_check()  -> tries 9 variants (heads x activation) on Phase-1 val tokens, saves best to VARIANT_PATH
STEP 2:         build_arch1(path) -> (bidirectional hidden-state backbone, 768) for the NER pipeline
"""
import itertools, json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch1_variant.json"
DEFAULT_V = {"H": 12, "act": "gelu"}
VOCAB, D, NL, DFF, MAXPOS = 40000, 768, 8, 3072, 1024


def act_fn(x, name):
    if name == "gelu": return F.gelu(x)
    if name == "gelu_tanh": return F.gelu(x, approximate="tanh")
    return F.relu(x)


class Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(D, D) for _ in range(4))

    def forward(self, x, mask, H):
        B, T, _ = x.shape
        sp = lambda t: t.view(B, T, H, D // H).transpose(1, 2)
        o = F.scaled_dot_product_attention(sp(self.q_proj(x)), sp(self.k_proj(x)), sp(self.v_proj(x)), attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, D))


class MLP(nn.Module):
    def __init__(self):
        super().__init__(); self.fc1 = nn.Linear(D, DFF); self.fc2 = nn.Linear(DFF, D)

    def forward(self, x, V): return self.fc2(act_fn(self.fc1(x), V["act"]))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D); self.attn = Attn(); self.norm2 = nn.LayerNorm(D); self.mlp = MLP()

    def forward(self, x, mask, V):
        x = x + self.attn(self.norm1(x), mask, V["H"])
        return x + self.mlp(self.norm2(x), V)


class Arch1(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb_ = None
        self.token_emb = nn.Embedding(VOCAB, D); self.pos_emb = nn.Embedding(MAXPOS, D)
        self.blocks = nn.ModuleList([Block() for _ in range(NL)])
        self.final_norm = nn.LayerNorm(D)
        self.lm_head = nn.Linear(D, VOCAB, bias=False); self.lm_head.weight = self.token_emb.weight  # tied
        self.V = dict(DEFAULT_V)

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev = ids.device
        if causal: m = torch.tril(torch.ones(T, T, dtype=torch.bool, device=dev))[None, None]
        else: m = torch.ones(1, 1, T, T, dtype=torch.bool, device=dev)
        if pad_mask is not None: m = m & pad_mask.bool()[:, None, None, :]
        x = self.token_emb(ids) + self.pos_emb(torch.arange(T, device=dev))[None]
        for blk in self.blocks: x = blk(x, m, self.V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = {k: v for k, v in ck["model"].items() if not k.endswith(".attn.mask")}  # causal-mask buffers; we build our own
    m = Arch1(); m.load_state_dict(sd, strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=4, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best:", ck.get("best"), "| step:", ck.get("step"), "| n_params:", ck.get("n_params"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= VOCAB: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    x = torch.stack([b for b in batches if len(b) == seq_len + 1]).to(dev)
    res = []
    for H, act in itertools.product([12, 8, 16], ["gelu", "gelu_tanh", "relu"]):
        m.V = {"H": H, "act": act}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V))); print(f"{loss:8.4f}  {m.V}")
    res.sort(key=lambda r: r[0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0])
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH)


def build_arch1(path=CKPT, causal=False):
    m, _ = load_model(path, "cpu")
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    return NERBackbone(m, causal=causal), D


if __name__ == "__main__":
    run_check()

checkpoint best: 4.100625859101613 | step: 10000 | n_params: 88210944
  4.1079  {'H': 12, 'act': 'gelu'}
  4.1077  {'H': 12, 'act': 'gelu_tanh'}
  4.7651  {'H': 12, 'act': 'relu'}
  3.8191  {'H': 8, 'act': 'gelu'}
  3.8190  {'H': 8, 'act': 'gelu_tanh'}
  4.3196  {'H': 8, 'act': 'relu'}
  4.0962  {'H': 16, 'act': 'gelu'}
  4.0960  {'H': 16, 'act': 'gelu_tanh'}
  4.7545  {'H': 16, 'act': 'relu'}

BEST: (np.float64(3.819012482961019), {'H': 8, 'act': 'gelu_tanh'}) | runner-up loss: 3.819054424762726
saved /kaggle/working/arch1_variant.json


In [17]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except ImportError:  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    tag = a.arch + (f"_f{a.fold}" if a.fold >= 0 else "")
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch])
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

models averaged: 5 | docs: 35 | entities: 4155


In [18]:
import sys, gc, torch
for f in range(5):
    sys.argv = ["x", "--arch", "arch1", "--fold", str(f), "--epochs", "6",
                "--lr_bb", "5e-5", "--role_boost", "2.0"]
    main()
    gc.collect(); torch.cuda.empty_cache()
sys.argv = ["x", "--arch", "arch1", "--ensemble"]
main()      # -> submission_arch1_ens.zip

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891


/tmp/ipykernel_48/2890683536.py:282: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")


ep0 loss 0.9265 val strict macroF1 0.0447
   APP        P=0.000 R=0.000 F1=0.000
   RESP       P=0.000 R=0.000 F1=0.000
   A.COUNSEL  P=0.000 R=0.000 F1=0.000
   R.COUNSEL  P=0.000 R=0.000 F1=0.000
   JUDGE      P=0.000 R=0.000 F1=0.000
   WIT        P=0.000 R=0.000 F1=0.000
   AUTH       P=0.000 R=0.000 F1=0.000
   COURT      P=0.578 R=0.491 F1=0.531
   STAT       P=0.000 R=0.000 F1=0.000
   PREC       P=0.000 R=0.000 F1=0.000
   DATE       P=0.017 R=0.003 F1=0.006
   CASENO     P=0.000 R=0.000 F1=0.000
ep1 loss 0.5228 val strict macroF1 0.1217
   APP        P=0.054 R=0.084 F1=0.066
   RESP       P=0.082 R=0.059 F1=0.068
   A.COUNSEL  P=0.000 R=0.000 F1=0.000
   R.COUNSEL  P=0.017 R=0.095 F1=0.028
   JUDGE      P=0.000 R=0.000 F1=0.000
   WIT        P=0.000 R=0.000 F1=0.000
   AUTH       P=0.195 R=0.174 F1=0.184
   COURT      P=0.437 R=0.695 F1=0.537
   STAT       P=0.276 R=0.446 F1=0.341
   PREC       P=0.071 R=0.012 F1=0.021
   DATE       P=0.152 R=0.369 F1=0.215
   CASENO     P=0.0

In [19]:
import torch
p = "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt"
ck = torch.load(p, map_location="cpu", weights_only=False)
print(type(ck))
if isinstance(ck, dict):
    print("top-level keys:", list(ck.keys())[:20])
    for k in ("cfg", "config", "args", "model_args", "best", "best_val", "step", "n_params"):
        if k in ck: print(k, "=", ck[k])
# SWA checkpoints often store averaged weights under a different key
for key in ("swa", "swa_model", "ema", "model_swa", "model", "state_dict"):
    if isinstance(ck, dict) and key in ck and isinstance(ck[key], dict):
        print(f"candidate weights key: {key} ({len(ck[key])} tensors)")
sd = ck.get("model", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck.state_dict()
print("num tensors:", len(sd))
for k in list(sd.keys())[:30]:
    print(k, tuple(sd[k].shape))
print("...")
for k in list(sd.keys())[-8:]:
    print(k, tuple(sd[k].shape))

<class 'dict'>
top-level keys: ['model', 'step', 'best', 'cfg', 'n_params']
cfg = {'BLOCK': 1024, 'MICRO_BATCH': 2, 'GRAD_ACCUM': 8, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'W': 256, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}
best = 2.3814771915475528
step = 10000
n_params = 242501632
candidate weights key: model (260 tensors)
num tensors: 260
token_emb.weight (40000, 1024)
blocks.0.norm1.weight (1024,)
blocks.0.norm1.bias (1024,)
blocks.0.attn.q_proj.weight (1024, 1024)
blocks.0.attn.q_proj.bias (1024,)
blocks.0.attn.k_proj.weight (1024, 1024)
blocks.0.attn.k_proj.bias (1024,)
blocks.0.attn.v_proj.weight (1024, 1024)
blocks.0.attn.v_proj.bias (1024,)
blocks.0.attn.out_proj.weight (1024, 1024)
blocks.0.attn.out_proj.bias (1024,)
blocks.0.norm2.weight (1024,)
blocks.0.norm2.bias (1024,)
blocks.0.mlp.fc1.weight (4096, 1024)
blocks.0.mlp.fc1.bias (4096,)
blocks.0.mlp.fc2.weight (1024, 4096)
blocks.0.ml

In [20]:
"""
Arch3 (16 layers, d=1024, MLP 4096, sliding window W=256, no learned pos-emb, tied lm_head, 242.5M)

STEP 1 (once):  run_check()  -> tries 72 variants (position scheme x heads x activation x windowing) on Phase-1
                               val tokens; saves the lowest-loss one to VARIANT_PATH
STEP 2:         build_arch3(path, causal=False) -> (hidden-state backbone, 1024) for the NER pipeline
"""
import itertools, json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch3_variant.json"
DEFAULT_V = {"pos": "rope_half", "H": 16, "act": "gelu", "win": 1}
VOCAB, D, NL, DFF, WINDOW, ROPE_BASE = 40000, 1024, 16, 4096, 256, 10000.0


def act_fn(x, name):
    if name == "gelu": return F.gelu(x)
    if name == "gelu_tanh": return F.gelu(x, approximate="tanh")
    return F.relu(x)


def rope_cache(T, hd, device):
    inv = 1.0 / (ROPE_BASE ** (torch.arange(0, hd, 2, device=device).float() / hd))
    fr = torch.outer(torch.arange(T, device=device).float(), inv)
    return fr.cos(), fr.sin()


def apply_rope(x, cos, sin, interleaved):  # x [B,H,T,hd]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    if interleaved:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1)


def alibi_slopes(H):
    start = 2 ** (-8.0 / H)
    return torch.tensor([start ** (i + 1) for i in range(H)])


class Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(D, D) for _ in range(4))

    def forward(self, x, cos, sin, mask, V):
        B, T, _ = x.shape
        H = V["H"]
        sp = lambda t: t.view(B, T, H, D // H).transpose(1, 2)
        q, k, v = sp(self.q_proj(x)), sp(self.k_proj(x)), sp(self.v_proj(x))
        if V["pos"].startswith("rope"):
            il = V["pos"] == "rope_inter"
            q, k = apply_rope(q, cos, sin, il), apply_rope(k, cos, sin, il)
        if mask.dtype != torch.bool: mask = mask.to(q.dtype)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, D))


class MLP(nn.Module):
    def __init__(self):
        super().__init__(); self.fc1 = nn.Linear(D, DFF); self.fc2 = nn.Linear(DFF, D)

    def forward(self, x, V): return self.fc2(act_fn(self.fc1(x), V["act"]))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D); self.attn = Attn(); self.norm2 = nn.LayerNorm(D); self.mlp = MLP()

    def forward(self, x, cos, sin, mask, V):
        x = x + self.attn(self.norm1(x), cos, sin, mask, V)
        return x + self.mlp(self.norm2(x), V)


class Arch3(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB, D)
        self.blocks = nn.ModuleList([Block() for _ in range(NL)])
        self.final_norm = nn.LayerNorm(D)
        self.lm_head = nn.Linear(D, VOCAB, bias=False); self.lm_head.weight = self.token_emb.weight  # tied
        self.V = dict(DEFAULT_V)

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev, V = ids.device, self.V
        i = torch.arange(T, device=dev)[:, None]; j = torch.arange(T, device=dev)[None]
        dist = i - j
        allow = (dist >= 0) if causal else torch.ones(T, T, dtype=torch.bool, device=dev)
        if V["win"]: allow = allow & (dist.abs() < WINDOW)
        allow = allow[None, None]
        if pad_mask is not None: allow = allow & pad_mask.bool()[:, None, None, :]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # no fully-masked rows
        if V["pos"] == "alibi":
            bias = -alibi_slopes(V["H"]).to(dev)[None, :, None, None] * dist.abs()[None, None].float()
            mask = torch.where(allow, bias, torch.full((), float("-inf"), device=dev))
        else:
            mask = allow
        cos = sin = None
        if V["pos"].startswith("rope"): cos, sin = rope_cache(T, D // V["H"], dev)
        x = self.token_emb(ids)
        for blk in self.blocks: x = blk(x, cos, sin, mask, V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = Arch3(); m.load_state_dict(ck["model"], strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=3, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best:", ck.get("best"), "| step:", ck.get("step"), "| n_params:", ck.get("n_params"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= VOCAB: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    x = torch.stack([b for b in batches if len(b) == seq_len + 1]).to(dev)
    res = []
    for pos, H, act, win in itertools.product(["rope_half", "rope_inter", "alibi", "none"], [16, 8, 32],
                                              ["gelu", "gelu_tanh", "relu"], [1, 0]):
        m.V = {"pos": pos, "H": H, "act": act, "win": win}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V)))
    res.sort(key=lambda r: r[0])
    for l, v in res[:10]: print(f"{l:8.4f}  {v}")
    print("... worst:", res[-1][0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0], "| checkpoint best:", ck.get("best"))
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH)


def build_arch3(path=CKPT, causal=False):
    m, _ = load_model(path, "cpu")
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    return NERBackbone(m, causal=causal), D


if __name__ == "__main__":
    run_check()

checkpoint best: 2.3814771915475528 | step: 10000 | n_params: 242501632
  7.6880  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}
  7.6885  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 1}
  7.6903  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 0}
  7.6908  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 0}
  7.7329  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 1}
  7.7334  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 1}
  7.7356  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 0}
  7.7361  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 0}
  7.8017  {'pos': 'alibi', 'H': 8, 'act': 'gelu', 'win': 1}
  7.8022  {'pos': 'alibi', 'H': 8, 'act': 'gelu_tanh', 'win': 1}
... worst: 8.269876797993978

BEST: (np.float64(7.688036494784885), {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}) | runner-up loss: 7.688542789883083 | checkpoint best: 2.3814771915475528
saved /kaggle/working/arch3_variant.json


In [21]:

"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except ImportError:  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    tag = a.arch + (f"_f{a.fold}" if a.fold >= 0 else "")
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    model = NER(bb, d).to(dev)
    opt = torch.optim.AdamW([{"params": model.bb.parameters(), "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(model.state_dict(), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(model.state_dict(), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev))
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__":
    main()

models averaged: 5 | docs: 35 | entities: 5209


In [22]:
import sys
for extra in ([], ["--causal"]):
    sys.argv = ["x", "--arch", "arch3", "--fold", "0", "--epochs", "6",
                "--lr_bb", "3e-5", "--role_boost", "2.0"] + extra
    main()

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891


/tmp/ipykernel_48/115287853.py:283: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_48/115287853.py:295: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.6164 val strict macroF1 0.1977
   APP        P=0.033 R=0.096 F1=0.049
   RESP       P=0.028 R=0.035 F1=0.031
   A.COUNSEL  P=0.174 R=0.108 F1=0.133
   R.COUNSEL  P=0.161 R=0.429 F1=0.234
   JUDGE      P=0.043 R=0.067 F1=0.052
   WIT        P=0.098 R=0.211 F1=0.133
   AUTH       P=0.093 R=0.042 F1=0.058
   COURT      P=0.504 R=0.650 F1=0.567
   STAT       P=0.303 R=0.485 F1=0.373
   PREC       P=0.065 R=0.045 F1=0.053
   DATE       P=0.467 R=0.676 F1=0.552
   CASENO     P=0.106 R=0.192 F1=0.136
ep1 loss 0.2191 val strict macroF1 0.2915
   APP        P=0.067 R=0.229 F1=0.104
   RESP       P=0.061 R=0.059 F1=0.060
   A.COUNSEL  P=0.176 R=0.081 F1=0.111
   R.COUNSEL  P=0.277 R=0.310 F1=0.292
   JUDGE      P=0.076 R=0.117 F1=0.092
   WIT        P=0.182 R=0.316 F1=0.231
   AUTH       P=0.297 R=0.677 F1=0.413
   COURT      P=0.493 R=0.609 F1=0.545
   STAT       P=0.487 R=0.668 F1=0.563
   PREC       P=0.180 R=0.174 F1=0.177
   DATE       P=0.575 R=0.768 F1=0.658
   CASENO     P=0.1

In [23]:
run_check()

checkpoint best: 2.3814771915475528 | step: 10000 | n_params: 242501632
  7.6880  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}
  7.6885  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 1}
  7.6903  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 0}
  7.6908  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 0}
  7.7329  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 1}
  7.7334  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 1}
  7.7356  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 0}
  7.7361  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 0}
  7.8017  {'pos': 'alibi', 'H': 8, 'act': 'gelu', 'win': 1}
  7.8022  {'pos': 'alibi', 'H': 8, 'act': 'gelu_tanh', 'win': 1}
... worst: 8.269876797993978

BEST: (np.float64(7.688036494784885), {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}) | runner-up loss: 7.688542789883083 | checkpoint best: 2.3814771915475528
saved /kaggle/working/arch3_variant.json


In [24]:
import zipfile, json
z = zipfile.ZipFile("/kaggle/working/submission_arch4_ens.zip")
print(z.namelist())     # sirf ['prediction.json']
p = json.loads(z.read("prediction.json"))
test = json.load(open("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER/Test/fold3-text.json"))
print(len(p), "docs | offset mismatches:", sum(1 for d in p for s,e,l,t in p[d] if test[d][s:e] != t))

['prediction.json']
35 docs | offset mismatches: 0


In [25]:
"""
Arch3 (16 layers, d=1024, MLP 4096, sliding window W=256, no learned pos-emb, tied lm_head, 242.5M)

STEP 1 (once):  run_check()  -> tries 72 variants (position scheme x heads x activation x windowing) on Phase-1
                               val tokens; saves the lowest-loss one to VARIANT_PATH
STEP 2:         build_arch3(path, causal=False) -> (hidden-state backbone, 1024) for the NER pipeline
"""
import itertools, json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch3_variant.json"
DEFAULT_V = {"pos": "rope_half", "H": 16, "act": "gelu", "win": 1}
VOCAB, D, NL, DFF, WINDOW, ROPE_BASE = 40000, 1024, 16, 4096, 256, 10000.0


def act_fn(x, name):
    if name == "gelu": return F.gelu(x)
    if name == "gelu_tanh": return F.gelu(x, approximate="tanh")
    return F.relu(x)


def rope_cache(T, hd, device, base=ROPE_BASE):
    inv = 1.0 / (base ** (torch.arange(0, hd, 2, device=device).float() / hd))
    fr = torch.outer(torch.arange(T, device=device).float(), inv)
    return fr.cos(), fr.sin()


def apply_rope(x, cos, sin, interleaved):  # x [B,H,T,hd]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    if interleaved:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1)


def alibi_slopes(H):
    start = 2 ** (-8.0 / H)
    return torch.tensor([start ** (i + 1) for i in range(H)])


class Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(D, D) for _ in range(4))

    def forward(self, x, cos, sin, mask, V):
        B, T, _ = x.shape
        H = V["H"]
        sp = lambda t: t.view(B, T, H, D // H).transpose(1, 2)
        q, k, v = sp(self.q_proj(x)), sp(self.k_proj(x)), sp(self.v_proj(x))
        if V["pos"].startswith("rope"):
            il = V["pos"] == "rope_inter"
            q, k = apply_rope(q, cos, sin, il), apply_rope(k, cos, sin, il)
        if mask.dtype != torch.bool: mask = mask.to(q.dtype)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, D))


class MLP(nn.Module):
    def __init__(self):
        super().__init__(); self.fc1 = nn.Linear(D, DFF); self.fc2 = nn.Linear(DFF, D)

    def forward(self, x, V): return self.fc2(act_fn(self.fc1(x), V["act"]))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D); self.attn = Attn(); self.norm2 = nn.LayerNorm(D); self.mlp = MLP()

    def forward(self, x, cos, sin, mask, V):
        x = x + self.attn(self.norm1(x), cos, sin, mask, V)
        return x + self.mlp(self.norm2(x), V)


class Arch3(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB, D)
        self.blocks = nn.ModuleList([Block() for _ in range(NL)])
        self.final_norm = nn.LayerNorm(D)
        self.lm_head = nn.Linear(D, VOCAB, bias=False); self.lm_head.weight = self.token_emb.weight  # tied
        self.V = dict(DEFAULT_V)

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev, V = ids.device, self.V
        i = torch.arange(T, device=dev)[:, None]; j = torch.arange(T, device=dev)[None]
        dist = i - j
        allow = (dist >= 0) if causal else torch.ones(T, T, dtype=torch.bool, device=dev)
        if V["win"]: allow = allow & (dist.abs() < WINDOW)
        allow = allow[None, None]
        if pad_mask is not None: allow = allow & pad_mask.bool()[:, None, None, :]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # no fully-masked rows
        if V["pos"] == "alibi":
            bias = -alibi_slopes(V["H"]).to(dev)[None, :, None, None] * dist.abs()[None, None].float()
            mask = torch.where(allow, bias, torch.full((), float("-inf"), device=dev))
        else:
            mask = allow
        cos = sin = None
        if V["pos"].startswith("rope"): cos, sin = rope_cache(T, D // V["H"], dev, V.get("base", ROPE_BASE))
        x = self.token_emb(ids)
        for blk in self.blocks: x = blk(x, cos, sin, mask, V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = Arch3(); m.load_state_dict(ck["model"], strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=3, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best:", ck.get("best"), "| step:", ck.get("step"), "| n_params:", ck.get("n_params"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= VOCAB: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    x = torch.stack([b for b in batches if len(b) == seq_len + 1]).to(dev)
    res = []
    for pos, H, act, win in itertools.product(["rope_half", "rope_inter", "alibi", "none"], [16, 8, 32],
                                              ["gelu", "gelu_tanh", "relu"], [1, 0]):
        m.V = {"pos": pos, "H": H, "act": act, "win": win}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V)))
    res.sort(key=lambda r: r[0])
    for l, v in res[:10]: print(f"{l:8.4f}  {v}")
    print("... worst:", res[-1][0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0], "| checkpoint best:", ck.get("best"))
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH)


@torch.no_grad()
def run_diag(n=24, lens=(8, 32, 128, 512)):
    """Short-sequence diagnostic. If the best loss at len 8/32 is already ~7+, the BLOCK structure
    (heads/activation/norm) is wrong. If short is OK (<~4.5) but long gets bad, position/window scheme is wrong."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best:", ck.get("best"))
    arrs = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if os.path.exists(p):
            a = np.fromfile(p, dtype=np.uint16)
            arrs.append(a if a.max() < VOCAB else np.fromfile(p, dtype=np.uint32))
    for L in lens:
        xs = []
        for k in range(n):
            a = arrs[k % len(arrs)]; o = (k + 1) * (len(a) // (n + 2))
            xs.append(torch.from_numpy(a[o:o + L + 1].astype(np.int64)))
        x = torch.stack(xs).to(dev)
        res = []
        for pos, base, H, act in itertools.product(["rope_half", "rope_inter", "alibi", "none"], [10000.0, 1e6],
                                                   [16, 8, 32], ["gelu", "relu"]):
            if pos != "rope_half" and pos != "rope_inter" and base != 10000.0: continue
            m.V = {"pos": pos, "base": base, "H": H, "act": act, "win": 1}
            logits = m(x[:, :-1], causal=True).float()
            loss = F.cross_entropy(logits.reshape(-1, VOCAB), x[:, 1:].reshape(-1)).item()
            res.append((loss, dict(m.V)))
        res.sort(key=lambda r: r[0])
        print(f"len {L:4d}: best {res[0][0]:.3f} {res[0][1]} | 2nd {res[1][0]:.3f} | worst {res[-1][0]:.3f}")


def build_arch3(path=CKPT, causal=False):
    m, _ = load_model(path, "cpu")
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    return NERBackbone(m, causal=causal), D


if __name__ == "__main__":
    run_check()

checkpoint best: 2.3814771915475528 | step: 10000 | n_params: 242501632
  7.6880  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}
  7.6885  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 1}
  7.6903  {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 0}
  7.6908  {'pos': 'alibi', 'H': 32, 'act': 'gelu_tanh', 'win': 0}
  7.7329  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 1}
  7.7334  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 1}
  7.7356  {'pos': 'alibi', 'H': 16, 'act': 'gelu', 'win': 0}
  7.7361  {'pos': 'alibi', 'H': 16, 'act': 'gelu_tanh', 'win': 0}
  7.8017  {'pos': 'alibi', 'H': 8, 'act': 'gelu', 'win': 1}
  7.8022  {'pos': 'alibi', 'H': 8, 'act': 'gelu_tanh', 'win': 1}
... worst: 8.269876797993978

BEST: (np.float64(7.688036494784885), {'pos': 'alibi', 'H': 32, 'act': 'gelu', 'win': 1}) | runner-up loss: 7.688542789883083 | checkpoint best: 2.3814771915475528
saved /kaggle/working/arch3_variant.json


In [26]:
import glob
print(sorted(glob.glob("/kaggle/working/arch4_f*_best.pt")))       # 5 fold weights
print(len(glob.glob("/kaggle/working/em_arch4_f*_*.npy")), "emission files")   # 5 folds x 35 docs = 175

['/kaggle/working/arch4_f0_best.pt', '/kaggle/working/arch4_f1_best.pt', '/kaggle/working/arch4_f2_best.pt', '/kaggle/working/arch4_f3_best.pt', '/kaggle/working/arch4_f4_best.pt']
175 emission files


In [28]:
"""
Paste this in a cell AFTER the pipeline cell (it uses load_split, encode_doc, predict_doc, viterbi,
tags_to_spans, NER, build_backbone, CKPT, OUT, LABELS from the pipeline).

    analyze(arch="arch4", fold=0)

For every gold span in the fold's validation docs it says WHY it was not an exact match:
  exact                    -> counted correct
  wrong_label_same_span    -> right boundaries, wrong label        (fix: context/window, class confusion)
  right_label_bad_boundary -> right label, boundaries off          (fix: boundary snapping / span trimming)
  overlaps_other_label     -> overlaps predictions of other labels only
  missed                   -> nothing predicted there              (fix: class weights / more signal)
plus FP = predicted spans that match no gold span.
"""
from collections import defaultdict
import torch


def analyze(arch="arch4", fold=0, W=512, stride=384, n_examples=3):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    _, va = load_split(fold=fold, nfolds=5)
    bb, d = build_backbone(arch, CKPT[arch], causal=False)
    model = NER(bb, d).to(dev)
    model.load_state_dict(torch.load(f"{OUT}/{arch}_f{fold}_best.pt", map_location=dev))
    model.eval()

    cats = {l: defaultdict(int) for l in LABELS}
    fp = defaultdict(int)
    ex = defaultdict(list)
    for k, (text, ents) in va.items():
        ids, offs, _ = encode_doc(text)
        if len(ids) == 0:
            continue
        pred = tags_to_spans(viterbi(predict_doc(model, ids, W, stride, dev)), offs, text)
        gold = [(e[0], e[1], e[2]) for e in ents]
        pset, gset = set(pred), set(gold)
        for p in pred:
            if p not in gset:
                fp[p[2]] += 1
        for g in gold:
            s, e, l = g
            ov = [p for p in pred if p[0] < e and p[1] > s]
            if g in pset: c = "exact"
            elif any(p[0] == s and p[1] == e for p in ov): c = "wrong_label_same_span"
            elif any(p[2] == l for p in ov): c = "right_label_bad_boundary"
            elif ov: c = "overlaps_other_label"
            else: c = "missed"
            cats[l][c] += 1
            if c != "exact" and len(ex[(l, c)]) < n_examples:
                ex[(l, c)].append((text[s:e][:60], [(text[p[0]:p[1]][:60], p[2]) for p in ov][:2]))

    cols = ["exact", "wrong_label_same_span", "right_label_bad_boundary", "overlaps_other_label", "missed"]
    print(f"{'label':10s} {'gold':>5s} " + " ".join(f"{c[:11]:>11s}" for c in cols) + f" {'FP':>5s}")
    for l in LABELS:
        n = sum(cats[l].values())
        print(f"{l:10s} {n:5d} " + " ".join(f"{cats[l][c]:11d}" for c in cols) + f" {fp[l]:5d}")
    print("\nExamples (gold text -> overlapping predictions):")
    for (l, c), items in ex.items():
        if l in ("APP", "RESP", "WIT", "PREC", "AUTH", "A.COUNSEL", "R.COUNSEL"):
            for g, p in items:
                print(f"  [{l} | {c}] {g!r} -> {p}")

In [29]:
import shutil, glob
shutil.copy("/kaggle/working/arch4_f0_best.pt", "/kaggle/working/arch4_f0_best_lr2e5.pt")
import os; os.makedirs("/kaggle/working/em_f0_backup", exist_ok=True)
for p in glob.glob("/kaggle/working/em_arch4_f0_*.npy"):
    shutil.copy(p, "/kaggle/working/em_f0_backup/")

In [11]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, sys, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except ImportError:  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, only A,B trained. B starts at 0 => identical to base at step 0."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


LORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")


def apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS):
    for p in backbone.parameters(): p.requires_grad = False  # freeze everything in the Phase-1 model
    n = 0
    for name, mod in list(backbone.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(mod, cname, LoRALinear(child, r, alpha, dropout)); n += 1
    return n


def ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--method", default="full", choices=["full", "lora"], help="full fine-tuning or LoRA")
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    tag = a.arch + ("" if a.method == "full" else f"_{a.method}") + (f"_f{a.fold}" if a.fold >= 0 else "")
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    if a.method == "lora":
        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout)
        print(f"LoRA: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")
    model = NER(bb, d).to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")
    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev), strict=False)
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook
    main()

In [7]:
"""
Arch4 (hybrid sliding-window/global GQA, 201M) reconstruction from checkpoint shapes.

STEP 1 (once):   run_check()   -> tries 16 layout variants on Phase-1 val tokens, saves the best to VARIANT_PATH
STEP 2:          build_arch4(path) -> (bidirectional hidden-state backbone, 1024) for the NER pipeline
"""
import itertools, json, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch4_variant.json"
DEFAULT_V = {"gl": 0, "rope": 0, "swi": 0, "qkv": "cat"}


def rope_cache(T, hd, base, device):
    inv = 1.0 / (base ** (torch.arange(0, hd, 2, device=device).float() / hd))
    fr = torch.outer(torch.arange(T, device=device).float(), inv)
    return fr.cos(), fr.sin()


def apply_rope(x, cos, sin, interleaved):  # x [B,H,T,hd]
    cos, sin = cos[None, None], sin[None, None]
    if interleaved:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1)


class Attn(nn.Module):
    def __init__(self, d, nq, nkv):
        super().__init__()
        self.nq, self.nkv, self.hd = nq, nkv, d // nq
        self.qkv_proj = nn.Linear(d, (nq + 2 * nkv) * self.hd, bias=False)
        self.out_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin, mask, V):
        B, T, _ = x.shape
        nq, nkv, hd = self.nq, self.nkv, self.hd
        rep = nq // nkv
        qkv = self.qkv_proj(x)
        if V["qkv"] == "cat":
            q, k, v = qkv.split([nq * hd, nkv * hd, nkv * hd], -1)
            q, k, v = q.view(B, T, nq, hd), k.view(B, T, nkv, hd), v.view(B, T, nkv, hd)
        else:  # grouped per kv head: [q*rep, k, v]
            g = qkv.view(B, T, nkv, rep + 2, hd)
            q = g[:, :, :, :rep].reshape(B, T, nq, hd); k = g[:, :, :, rep]; v = g[:, :, :, rep + 1]
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        q = apply_rope(q, cos, sin, V["rope"]); k = apply_rope(k, cos, sin, V["rope"])
        k = k.repeat_interleave(rep, 1); v = v.repeat_interleave(rep, 1)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, -1))


class FFN(nn.Module):
    def __init__(self, d, dff):
        super().__init__()
        self.in_proj = nn.Linear(d, 2 * dff, bias=False)
        self.out_proj = nn.Linear(dff, d, bias=False)

    def forward(self, x, V):
        a, b = self.in_proj(x).chunk(2, -1)
        return self.out_proj(F.silu(a) * b if V["swi"] == 0 else F.silu(b) * a)


class Block(nn.Module):
    def __init__(self, d, nq, nkv, dff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d); self.attn = Attn(d, nq, nkv)
        self.norm2 = nn.LayerNorm(d); self.ffn = FFN(d, dff)

    def forward(self, x, cos, sin, mask, V):
        x = x + self.attn(self.norm1(x), cos, sin, mask, V)
        return x + self.ffn(self.norm2(x), V)


class Arch4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg["D_MODEL"]
        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], d)
        self.layers = nn.ModuleList([Block(d, cfg["N_Q_HEADS"], cfg["N_KV_HEADS"], cfg["D_FF"])
                                     for _ in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["VOCAB_SIZE"], bias=False)
        self.V = dict(DEFAULT_V)

    def is_global(self, i):
        ge = self.cfg["GLOBAL_EVERY"]
        return (i + 1) % ge == 0 if self.V["gl"] == 0 else i % ge == 0

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev = ids.device
        win = self.cfg["WINDOW"]
        i = torch.arange(T, device=dev)[:, None]; j = torch.arange(T, device=dev)[None]
        dist = i - j
        if causal: loc, glob = (dist >= 0) & (dist < win), dist >= 0
        else: loc, glob = dist.abs() < win, torch.ones(T, T, dtype=torch.bool, device=dev)
        loc, glob = loc[None, None], glob[None, None]
        if pad_mask is not None:
            key = pad_mask.bool()[:, None, None, :]
            loc, glob = loc & key, glob & key
        cos, sin = rope_cache(T, self.cfg["D_MODEL"] // self.cfg["N_Q_HEADS"], self.cfg["ROPE_BASE"], dev)
        x = self.token_embedding(ids)
        for n, blk in enumerate(self.layers):
            x = blk(x, cos, sin, glob if self.is_global(n) else loc, self.V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    """Bidirectional hidden-state view of the (causally pretrained) model."""
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = Arch4(ck["cfg"]); m.load_state_dict(ck["model"], strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=4, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best_val:", ck.get("best_val"), "| step:", ck.get("step"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= m.cfg["VOCAB_SIZE"]: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    batches = [b for b in batches if len(b) == seq_len + 1]
    x = torch.stack(batches).to(dev)
    res = []
    for gl, rope, swi, qkv in itertools.product([0, 1], [0, 1], [0, 1], ["cat", "grouped"]):
        m.V = {"gl": gl, "rope": rope, "swi": swi, "qkv": qkv}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V))); print(f"{loss:8.4f}  {m.V}")
    res.sort(key=lambda r: r[0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0])
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH, "(loss should be close to best_val; clearly lower than all others)")


def build_arch4(path=CKPT, causal=False):
    dev = "cpu"
    m, _ = load_model(path, dev)
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    m.lm_head = nn.Identity()  # not needed for NER; saves memory (state already loaded)
    return NERBackbone(m, causal=causal), m.cfg["D_MODEL"]


if __name__ == "__main__":
    run_check()

checkpoint best_val: 2.475979009747505 | step: 10000
  2.2604  {'gl': 0, 'rope': 0, 'swi': 0, 'qkv': 'cat'}
  7.4957  {'gl': 0, 'rope': 0, 'swi': 0, 'qkv': 'grouped'}
  4.7490  {'gl': 0, 'rope': 0, 'swi': 1, 'qkv': 'cat'}
  8.2485  {'gl': 0, 'rope': 0, 'swi': 1, 'qkv': 'grouped'}
  4.2593  {'gl': 0, 'rope': 1, 'swi': 0, 'qkv': 'cat'}
  7.6124  {'gl': 0, 'rope': 1, 'swi': 0, 'qkv': 'grouped'}
  6.2932  {'gl': 0, 'rope': 1, 'swi': 1, 'qkv': 'cat'}
  8.3156  {'gl': 0, 'rope': 1, 'swi': 1, 'qkv': 'grouped'}
  2.6234  {'gl': 1, 'rope': 0, 'swi': 0, 'qkv': 'cat'}
  7.4198  {'gl': 1, 'rope': 0, 'swi': 0, 'qkv': 'grouped'}
  5.3667  {'gl': 1, 'rope': 0, 'swi': 1, 'qkv': 'cat'}
  8.1301  {'gl': 1, 'rope': 0, 'swi': 1, 'qkv': 'grouped'}
  4.3116  {'gl': 1, 'rope': 1, 'swi': 0, 'qkv': 'cat'}
  7.5284  {'gl': 1, 'rope': 1, 'swi': 0, 'qkv': 'grouped'}
  6.3226  {'gl': 1, 'rope': 1, 'swi': 1, 'qkv': 'cat'}
  8.2477  {'gl': 1, 'rope': 1, 'swi': 1, 'qkv': 'grouped'}

BEST: (np.float64(2.26035137971242

In [10]:
"""
Arch4 (hybrid sliding-window/global GQA, 201M) reconstruction from checkpoint shapes.

STEP 1 (once):   run_check()   -> tries 16 layout variants on Phase-1 val tokens, saves the best to VARIANT_PATH
STEP 2:          build_arch4(path) -> (bidirectional hidden-state backbone, 1024) for the NER pipeline
"""
import itertools, json, os, sys
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CKPT = "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
VAL_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
VAL_BINS = ["sc_val.bin", "acts_val.bin", "tribunals_val_f.bin"]
VARIANT_PATH = "/kaggle/working/arch4_variant.json"
DEFAULT_V = {"gl": 0, "rope": 0, "swi": 0, "qkv": "cat"}


def rope_cache(T, hd, base, device):
    inv = 1.0 / (base ** (torch.arange(0, hd, 2, device=device).float() / hd))
    fr = torch.outer(torch.arange(T, device=device).float(), inv)
    return fr.cos(), fr.sin()


def apply_rope(x, cos, sin, interleaved):  # x [B,H,T,hd]
    cos, sin = cos[None, None], sin[None, None]
    if interleaved:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1)


class Attn(nn.Module):
    def __init__(self, d, nq, nkv):
        super().__init__()
        self.nq, self.nkv, self.hd = nq, nkv, d // nq
        self.qkv_proj = nn.Linear(d, (nq + 2 * nkv) * self.hd, bias=False)
        self.out_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin, mask, V):
        B, T, _ = x.shape
        nq, nkv, hd = self.nq, self.nkv, self.hd
        rep = nq // nkv
        qkv = self.qkv_proj(x)
        if V["qkv"] == "cat":
            q, k, v = qkv.split([nq * hd, nkv * hd, nkv * hd], -1)
            q, k, v = q.view(B, T, nq, hd), k.view(B, T, nkv, hd), v.view(B, T, nkv, hd)
        else:  # grouped per kv head: [q*rep, k, v]
            g = qkv.view(B, T, nkv, rep + 2, hd)
            q = g[:, :, :, :rep].reshape(B, T, nq, hd); k = g[:, :, :, rep]; v = g[:, :, :, rep + 1]
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        q = apply_rope(q, cos, sin, V["rope"]); k = apply_rope(k, cos, sin, V["rope"])
        k = k.repeat_interleave(rep, 1); v = v.repeat_interleave(rep, 1)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, -1))


class FFN(nn.Module):
    def __init__(self, d, dff):
        super().__init__()
        self.in_proj = nn.Linear(d, 2 * dff, bias=False)
        self.out_proj = nn.Linear(dff, d, bias=False)

    def forward(self, x, V):
        a, b = self.in_proj(x).chunk(2, -1)
        return self.out_proj(F.silu(a) * b if V["swi"] == 0 else F.silu(b) * a)


class Block(nn.Module):
    def __init__(self, d, nq, nkv, dff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d); self.attn = Attn(d, nq, nkv)
        self.norm2 = nn.LayerNorm(d); self.ffn = FFN(d, dff)

    def forward(self, x, cos, sin, mask, V):
        x = x + self.attn(self.norm1(x), cos, sin, mask, V)
        return x + self.ffn(self.norm2(x), V)


class Arch4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg["D_MODEL"]
        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], d)
        self.layers = nn.ModuleList([Block(d, cfg["N_Q_HEADS"], cfg["N_KV_HEADS"], cfg["D_FF"])
                                     for _ in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["VOCAB_SIZE"], bias=False)
        self.V = dict(DEFAULT_V)

    def is_global(self, i):
        ge = self.cfg["GLOBAL_EVERY"]
        return (i + 1) % ge == 0 if self.V["gl"] == 0 else i % ge == 0

    def forward(self, ids, pad_mask=None, causal=True, return_hidden=False):
        B, T = ids.shape
        dev = ids.device
        win = self.cfg["WINDOW"]
        i = torch.arange(T, device=dev)[:, None]; j = torch.arange(T, device=dev)[None]
        dist = i - j
        if causal: loc, glob = (dist >= 0) & (dist < win), dist >= 0
        else: loc, glob = dist.abs() < win, torch.ones(T, T, dtype=torch.bool, device=dev)
        loc, glob = loc[None, None], glob[None, None]
        if pad_mask is not None:
            key = pad_mask.bool()[:, None, None, :]
            loc, glob = loc & key, glob & key
        cos, sin = rope_cache(T, self.cfg["D_MODEL"] // self.cfg["N_Q_HEADS"], self.cfg["ROPE_BASE"], dev)
        x = self.token_embedding(ids)
        for n, blk in enumerate(self.layers):
            x = blk(x, cos, sin, glob if self.is_global(n) else loc, self.V)
        x = self.final_norm(x)
        return x if return_hidden else self.lm_head(x)


class NERBackbone(nn.Module):
    """Bidirectional hidden-state view of the (causally pretrained) model."""
    def __init__(self, model, causal=False):
        super().__init__(); self.m, self.causal = model, causal

    def forward(self, ids, mask):
        return self.m(ids, pad_mask=mask, causal=self.causal, return_hidden=True)


def load_model(path=CKPT, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = Arch4(ck["cfg"]); m.load_state_dict(ck["model"], strict=True)
    return m.to(device), ck


@torch.no_grad()
def run_check(n_seq=4, seq_len=1024):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m, ck = load_model(device=dev); m.eval()
    print("checkpoint best_val:", ck.get("best_val"), "| step:", ck.get("step"))
    batches = []
    for f in VAL_BINS:
        p = os.path.join(VAL_DIR, f)
        if not os.path.exists(p): continue
        arr = np.fromfile(p, dtype=np.uint16)
        if arr.max() >= m.cfg["VOCAB_SIZE"]: arr = np.fromfile(p, dtype=np.uint32)
        for s in range(n_seq):
            o = s * (len(arr) // (n_seq + 1))
            batches.append(torch.from_numpy(arr[o:o + seq_len + 1].astype(np.int64)))
    batches = [b for b in batches if len(b) == seq_len + 1]
    x = torch.stack(batches).to(dev)
    res = []
    for gl, rope, swi, qkv in itertools.product([0, 1], [0, 1], [0, 1], ["cat", "grouped"]):
        m.V = {"gl": gl, "rope": rope, "swi": swi, "qkv": qkv}
        loss = np.mean([F.cross_entropy(m(x[i:i + 1, :-1], causal=True)[0].float(), x[i, 1:]).item()
                        for i in range(len(x))])
        res.append((loss, dict(m.V))); print(f"{loss:8.4f}  {m.V}")
    res.sort(key=lambda r: r[0])
    print("\nBEST:", res[0], "| runner-up loss:", res[1][0])
    json.dump(res[0][1], open(VARIANT_PATH, "w"))
    print("saved", VARIANT_PATH, "(loss should be close to best_val; clearly lower than all others)")


def build_arch4(path=CKPT, causal=False):
    dev = "cpu"
    m, _ = load_model(path, dev)
    if os.path.exists(VARIANT_PATH): m.V = json.load(open(VARIANT_PATH))
    else: print("WARNING: no variant file; run run_check() first. Using defaults.")
    m.lm_head = nn.Identity()  # not needed for NER; saves memory (state already loaded)
    return NERBackbone(m, causal=causal), m.cfg["D_MODEL"]


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # not auto-run in notebooks
    run_check()

In [13]:
import sys, os
sys.modules.pop("arch4_backbone", None)
if os.path.exists("/kaggle/working/arch4_backbone.py"):
    os.remove("/kaggle/working/arch4_backbone.py")
print("build_arch4 in notebook:", "build_arch4" in globals())     # True aana chahiye

build_arch4 in notebook: True


In [14]:
sys.argv = ["x", "--arch", "arch4", "--fold", "0", "--epochs", "8",
            "--method", "lora", "--rank", "16", "--alpha", "32",
            "--lr_bb", "3e-4", "--role_boost", "2.0"]
main()

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891
LoRA: wrapped 48 linear layers | rank 16 alpha 32.0
trainable params: 4.37M / 205.81M total (2.12%)


/tmp/ipykernel_49/1178580532.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_49/1178580532.py:337: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.6708 val strict macroF1 0.3058
   APP        P=0.103 R=0.072 F1=0.085
   RESP       P=0.039 R=0.035 F1=0.037
   A.COUNSEL  P=0.192 R=0.135 F1=0.159
   R.COUNSEL  P=0.191 R=0.500 F1=0.276
   JUDGE      P=0.343 R=0.383 F1=0.362
   WIT        P=0.044 R=0.211 F1=0.073
   AUTH       P=0.234 R=0.240 F1=0.237
   COURT      P=0.537 R=0.655 F1=0.590
   STAT       P=0.507 R=0.632 F1=0.562
   PREC       P=0.025 R=0.063 F1=0.035
   DATE       P=0.712 R=0.901 F1=0.795
   CASENO     P=0.360 R=0.629 F1=0.458
ep1 loss 0.1516 val strict macroF1 0.4320
   APP        P=0.237 R=0.169 F1=0.197
   RESP       P=0.148 R=0.141 F1=0.145
   A.COUNSEL  P=0.167 R=0.081 F1=0.109
   R.COUNSEL  P=0.345 R=0.714 F1=0.465
   JUDGE      P=0.415 R=0.650 F1=0.506
   WIT        P=0.067 R=0.474 F1=0.117
   AUTH       P=0.446 R=0.617 F1=0.518
   COURT      P=0.618 R=0.764 F1=0.683
   STAT       P=0.608 R=0.762 F1=0.676
   PREC       P=0.263 R=0.338 F1=0.296
   DATE       P=0.871 R=0.942 F1=0.905
   CASENO     P=0.5

In [15]:
import sys, gc, torch
for f in range(5):
    sys.argv = ["x", "--arch", "arch4", "--fold", str(f), "--epochs", "6",
                "--method", "lora", "--rank", "16", "--alpha", "32",
                "--lr_bb", "3e-4", "--role_boost", "2.0"]
    main()
    gc.collect(); torch.cuda.empty_cache()

sys.argv = ["x", "--arch", "arch4", "--method", "lora", "--ensemble"]
main()      # -> submission_arch4_lora_ens.zip

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891
LoRA: wrapped 48 linear layers | rank 16 alpha 32.0
trainable params: 4.37M / 205.81M total (2.12%)


/tmp/ipykernel_49/1178580532.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_49/1178580532.py:337: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.6157 val strict macroF1 0.3129
   APP        P=0.077 R=0.048 F1=0.059
   RESP       P=0.027 R=0.024 F1=0.025
   A.COUNSEL  P=0.241 R=0.378 F1=0.295
   R.COUNSEL  P=0.221 R=0.405 F1=0.286
   JUDGE      P=0.343 R=0.383 F1=0.362
   WIT        P=0.036 R=0.211 F1=0.062
   AUTH       P=0.247 R=0.234 F1=0.240
   COURT      P=0.482 R=0.595 F1=0.533
   STAT       P=0.503 R=0.583 F1=0.540
   PREC       P=0.020 R=0.054 F1=0.029
   DATE       P=0.762 R=0.894 F1=0.823
   CASENO     P=0.411 R=0.647 F1=0.502
ep1 loss 0.1442 val strict macroF1 0.4197
   APP        P=0.304 R=0.205 F1=0.245
   RESP       P=0.144 R=0.153 F1=0.149
   A.COUNSEL  P=0.000 R=0.000 F1=0.000
   R.COUNSEL  P=0.281 R=0.595 F1=0.382
   JUDGE      P=0.351 R=0.550 F1=0.429
   WIT        P=0.100 R=0.421 F1=0.162
   AUTH       P=0.458 R=0.683 F1=0.548
   COURT      P=0.631 R=0.786 F1=0.700
   STAT       P=0.604 R=0.769 F1=0.676
   PREC       P=0.236 R=0.296 F1=0.263
   DATE       P=0.862 R=0.935 F1=0.897
   CASENO     P=0.5

In [16]:
print("LoRMAPlusLinear" in globals())     # True aana chahiye

False


In [17]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, sys, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except (ImportError, AttributeError):  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, only A,B trained. B starts at 0 => identical to base at step 0."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


class LoRMAPlusLinear(nn.Module):
    """LoRMA+ (Low-Rank Multiplicative Adaptation, additive inflation):  W = (I + (alpha/r) B A) W0.
    With h = W0 x (frozen layer output):  y = h + (alpha/r) * B(A h).  A:(r,out)  B:(out,r), B=0 at init => identity => y = W0 x."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + (h @ self.A.t() @ self.B.t()) * self.scale


LORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")


def apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS, kind="lora"):
    for p in backbone.parameters(): p.requires_grad = False  # freeze everything in the Phase-1 model
    n = 0
    for name, mod in list(backbone.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                cls = LoRMAPlusLinear if kind == "lorma" else LoRALinear
                setattr(mod, cname, cls(child, r, alpha, dropout)); n += 1
    return n


def ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--method", default="full", choices=["full", "lora", "lorma"], help="full fine-tuning, LoRA, or LoRMA+ (multiplicative, additive inflation)")
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    tag = a.arch + ("" if a.method == "full" else f"_{a.method}") + (f"_f{a.fold}" if a.fold >= 0 else "")
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    if a.method in ("lora", "lorma"):
        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout, kind=a.method)
        print(f"{a.method.upper()}: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")
    model = NER(bb, d).to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")
    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            loss = F.cross_entropy(lg.float().view(-1, C), lab.view(-1), weight=cw, ignore_index=-100)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        if a.final_full:
            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev), strict=False)
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook
    main()

In [18]:
print("LoRMAPlusLinear" in globals())     # ab True aana chahiye
print("build_arch4" in globals() or "arch4_backbone" in sys.modules)   # True aana chahiye

True
True


In [19]:
import sys
sys.argv = ["x", "--arch", "arch4", "--fold", "0", "--epochs", "8",
            "--method", "lorma", "--rank", "16", "--alpha", "32",
            "--lr_bb", "3e-4", "--role_boost", "2.0"]
main()

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891
LORMA: wrapped 48 linear layers | rank 16 alpha 32.0
trainable params: 5.20M / 206.65M total (2.52%)


/tmp/ipykernel_49/1698594433.py:342: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_49/1698594433.py:354: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


ep0 loss 0.6001 val strict macroF1 0.3515
   APP        P=0.164 R=0.349 F1=0.223
   RESP       P=0.062 R=0.024 F1=0.034
   A.COUNSEL  P=0.333 R=0.486 F1=0.396
   R.COUNSEL  P=0.146 R=0.167 F1=0.156
   JUDGE      P=0.231 R=0.400 F1=0.293
   WIT        P=0.094 R=0.421 F1=0.154
   AUTH       P=0.236 R=0.377 F1=0.290
   COURT      P=0.551 R=0.732 F1=0.629
   STAT       P=0.481 R=0.700 F1=0.570
   PREC       P=0.145 R=0.204 F1=0.169
   DATE       P=0.685 R=0.860 F1=0.762
   CASENO     P=0.473 R=0.635 F1=0.542
ep1 loss 0.1409 val strict macroF1 0.4393
   APP        P=0.190 R=0.133 F1=0.156
   RESP       P=0.123 R=0.094 F1=0.107
   A.COUNSEL  P=0.375 R=0.324 F1=0.348
   R.COUNSEL  P=0.333 R=0.571 F1=0.421
   JUDGE      P=0.461 R=0.583 F1=0.515
   WIT        P=0.045 R=0.263 F1=0.078
   AUTH       P=0.402 R=0.665 F1=0.501
   COURT      P=0.590 R=0.773 F1=0.669
   STAT       P=0.681 R=0.814 F1=0.742
   PREC       P=0.283 R=0.350 F1=0.313
   DATE       P=0.869 R=0.857 F1=0.863
   CASENO     P=0.4

In [20]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, sys, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except (ImportError, AttributeError):  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, only A,B trained. B starts at 0 => identical to base at step 0."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


class LoRMAPlusLinear(nn.Module):
    """LoRMA+ (Low-Rank Multiplicative Adaptation, additive inflation):  W = (I + (alpha/r) B A) W0.
    With h = W0 x (frozen layer output):  y = h + (alpha/r) * B(A h).  A:(r,out)  B:(out,r), B=0 at init => identity => y = W0 x."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + (h @ self.A.t() @ self.B.t()) * self.scale


LORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")


def apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS, kind="lora"):
    for p in backbone.parameters(): p.requires_grad = False  # freeze everything in the Phase-1 model
    n = 0
    for name, mod in list(backbone.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                cls = LoRMAPlusLinear if kind == "lorma" else LoRALinear
                setattr(mod, cname, cls(child, r, alpha, dropout)); n += 1
    return n


def ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


def build_teacher(arch, method, fold, dev, rank, alpha, teacher_ckpt=None, teacher_dir=OUT):
    """Load a frozen, fold-matched NER teacher. Explicit --kd_teacher_ckpt overrides directory naming."""
    bb_t, d_t = build_backbone(arch, CKPT[arch], causal=False)
    if method in ("lora", "lorma"):
        apply_lora(bb_t, rank, alpha, 0.0, kind=method)
    t = NER(bb_t, d_t)
    path = teacher_ckpt or os.path.join(
        teacher_dir, f"{arch}{'' if method == 'full' else '_' + method}_f{fold}_best.pt"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Pass --kd_teacher_ckpt /path/to/teacher.pt "
            "or set --kd_teacher_dir to the directory containing fold teacher checkpoints."
        )
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
        state = state["model"]
    res = t.load_state_dict(state, strict=False)
    if res.unexpected_keys:
        raise RuntimeError(f"Teacher checkpoint has unexpected keys: {res.unexpected_keys[:10]}")
    # Missing frozen base weights can be expected when the file stores only LoRA/head trainable weights.
    t = t.to(dev).eval()
    for p in t.parameters():
        p.requires_grad = False
    print(f"teacher: {path} | params {sum(p.numel() for p in t.parameters())/1e6:.1f}M")
    print(f"teacher checkpoint load: missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}")
    return t


class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--method", default="full", choices=["full", "lora", "lorma"], help="full fine-tuning, LoRA, or LoRMA+ (multiplicative, additive inflation)")
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--kd", action="store_true", help="knowledge distillation from a fine-tuned teacher")
    ap.add_argument("--kd_teacher_method", default="lora", choices=["full", "lora", "lorma"])
    ap.add_argument("--kd_t_rank", type=int, default=16); ap.add_argument("--kd_t_alpha", type=float, default=32.0)
    ap.add_argument("--kd_T", type=float, default=2.0, help="softmax temperature")
    ap.add_argument("--kd_alpha", type=float, default=0.5, help="weight of hard-label CE; (1-alpha) goes to the KD term")
    ap.add_argument("--kd_teacher_ckpt", default="", help="explicit path to the fold-matched fine-tuned teacher checkpoint")
    ap.add_argument("--kd_teacher_dir", default=OUT, help="directory containing <arch>_<teacher_method>_f<fold>_best.pt")
    ap.add_argument("--student_layers", default="0,1,2,6,7,8", help="student backbone layer indices to retain")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    if a.kd and a.fold < 0:
        ap.error("KD requires --fold 0..4 so the student uses the matching fold teacher and validation split.")
    if a.kd and not a.student_layers:
        ap.error("KD student requires --student_layers; recommended Arch4 subset: 0,1,2,6,7,8.")
    n_keep = len([x for x in a.student_layers.split(",") if x.strip()]) if a.student_layers else 0
    tag = (a.arch + ("" if a.method == "full" else f"_{a.method}") + ("_kd" if a.kd else "")
           + (f"_s{n_keep}" if n_keep else "") + (f"_f{a.fold}" if a.fold >= 0 else ""))
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    # Build the full pretrained backbone, then retain only the selected layers for the student.
    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    idx = []
    if a.student_layers:
        idx = [int(x.strip()) for x in a.student_layers.split(",") if x.strip()]
        if len(set(idx)) != len(idx):
            raise ValueError(f"Duplicate student layer indices: {idx}")
        inner = getattr(bb, "m", None)
        if inner is None or not hasattr(inner, "layers"):
            raise AttributeError("Student layer surgery expects backbone.m.layers; adapt this block to your Arch4 wrapper.")
        n_layers = len(inner.layers)
        if any(i < 0 or i >= n_layers for i in idx):
            raise IndexError(f"student_layers={idx}, but backbone has {n_layers} layers.")
        inner.layers = nn.ModuleList([inner.layers[i] for i in idx])
        print(f"student: kept layers {idx} from {n_layers} | student backbone layers={len(inner.layers)}")

    # Teacher is built independently with the FULL backbone; never apply student layer pruning to it.
    teacher = build_teacher(
        a.arch, a.kd_teacher_method, a.fold, dev, a.kd_t_rank, a.kd_t_alpha,
        teacher_ckpt=(a.kd_teacher_ckpt or None), teacher_dir=a.kd_teacher_dir
    ) if a.kd else None
    if a.method in ("lora", "lorma"):
        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout, kind=a.method)
        print(f"{a.method.upper()}: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")
    model = NER(bb, d).to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")
    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            ce_loss = F.cross_entropy(lg.float().reshape(-1, C), lab.reshape(-1), weight=cw, ignore_index=-100)
            kd_loss = torch.zeros((), device=dev)
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                    tl = teacher(ids, msk).float()
                tok_mask = (lab != -100) & (msk != 0)
                if tok_mask.any():
                    T = a.kd_T
                    # KL(teacher || student), computed only on supervised/non-padding tokens.
                    kd_loss = F.kl_div(
                        F.log_softmax(lg.float()[tok_mask] / T, dim=-1),
                        F.softmax(tl[tok_mask] / T, dim=-1),
                        reduction="batchmean"
                    ) * (T * T)
                    loss = a.kd_alpha * ce_loss + (1.0 - a.kd_alpha) * kd_loss
                else:
                    loss = ce_loss
            else:
                loss = ce_loss
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if teacher is not None:
                # Running CE/KD means are useful for checking loss-scale imbalance.
                pass
        if a.final_full:
            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev), strict=False)
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook
    main()

In [22]:
import sys

sys.argv = [
    "x",
    "--arch", "arch4",
    "--fold", "0",
    "--nfolds", "5",
    "--epochs", "8",

    # Student fine-tuning
    "--method", "lorma",
    "--rank", "16",
    "--alpha", "32",
    "--lr_bb", "3e-4",
    "--lr_head", "1e-3",
    "--role_boost", "2.0",

    # Knowledge Distillation
    "--kd",
    "--kd_teacher_method", "lora",
    "--kd_t_rank", "16",
    "--kd_t_alpha", "32",
    "--kd_T", "2.0",
    "--kd_alpha", "0.5",

    # Six-layer student
    "--student_layers", "0,1,2,6,7,8",

    # Window configuration
    "--W", "512",
    "--stride", "384",
    "--bs", "8"
]

main()

WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891
student: kept layers [0, 1, 2, 6, 7, 8] from 12 | student backbone layers=6
teacher: /kaggle/working/arch4_lora_f0_best.pt | params 205.8M
teacher checkpoint load: missing=99, unexpected=0
LORMA: wrapped 24 linear layers | rank 16 alpha 32.0
trainable params: 3.14M / 124.34M total (2.52%)


/tmp/ipykernel_49/2905831559.py:405: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
/tmp/ipykernel_49/2905831559.py:435: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


ep0 loss 1.2982 val strict macroF1 0.3083
   APP        P=0.074 R=0.084 F1=0.079
   RESP       P=0.082 R=0.047 F1=0.060
   A.COUNSEL  P=0.082 R=0.351 F1=0.133
   R.COUNSEL  P=0.000 R=0.000 F1=0.000
   JUDGE      P=0.159 R=0.233 F1=0.189
   WIT        P=0.000 R=0.000 F1=0.000
   AUTH       P=0.344 R=0.269 F1=0.302
   COURT      P=0.601 R=0.755 F1=0.669
   STAT       P=0.612 R=0.668 F1=0.639
   PREC       P=0.215 R=0.237 F1=0.225
   DATE       P=0.806 R=0.877 F1=0.840
   CASENO     P=0.510 R=0.629 F1=0.563
ep1 loss 0.1768 val strict macroF1 0.4392
   APP        P=0.235 R=0.386 F1=0.292
   RESP       P=0.175 R=0.082 F1=0.112
   A.COUNSEL  P=0.324 R=0.595 F1=0.419
   R.COUNSEL  P=0.538 R=0.333 F1=0.412
   JUDGE      P=0.310 R=0.433 F1=0.361
   WIT        P=0.091 R=0.053 F1=0.067
   AUTH       P=0.470 R=0.563 F1=0.512
   COURT      P=0.684 R=0.818 F1=0.745
   STAT       P=0.601 R=0.739 F1=0.663
   PREC       P=0.331 R=0.335 F1=0.333
   DATE       P=0.835 R=0.932 F1=0.881
   CASENO     P=0.3

In [23]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, sys, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except (ImportError, AttributeError):  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, only A,B trained. B starts at 0 => identical to base at step 0."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


class LoRMAPlusLinear(nn.Module):
    """LoRMA+ (Low-Rank Multiplicative Adaptation, additive inflation):  W = (I + (alpha/r) B A) W0.
    With h = W0 x (frozen layer output):  y = h + (alpha/r) * B(A h).  A:(r,out)  B:(out,r), B=0 at init => identity => y = W0 x."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + (h @ self.A.t() @ self.B.t()) * self.scale


LORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")


def apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS, kind="lora"):
    for p in backbone.parameters(): p.requires_grad = False  # freeze everything in the Phase-1 model
    n = 0
    for name, mod in list(backbone.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                cls = LoRMAPlusLinear if kind == "lorma" else LoRALinear
                setattr(mod, cname, cls(child, r, alpha, dropout)); n += 1
    return n


def ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


def build_teacher(arch, method, fold, dev, rank, alpha, teacher_ckpt=None, teacher_dir=OUT):
    """Load a frozen, fold-matched NER teacher. Explicit --kd_teacher_ckpt overrides directory naming."""
    bb_t, d_t = build_backbone(arch, CKPT[arch], causal=False)
    if method in ("lora", "lorma"):
        apply_lora(bb_t, rank, alpha, 0.0, kind=method)
    t = NER(bb_t, d_t)
    path = teacher_ckpt or os.path.join(
        teacher_dir, f"{arch}{'' if method == 'full' else '_' + method}_f{fold}_best.pt"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Pass --kd_teacher_ckpt /path/to/teacher.pt "
            "or set --kd_teacher_dir to the directory containing fold teacher checkpoints."
        )
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
        state = state["model"]
    res = t.load_state_dict(state, strict=False)
    if res.unexpected_keys:
        raise RuntimeError(f"Teacher checkpoint has unexpected keys: {res.unexpected_keys[:10]}")
    # Missing frozen base weights can be expected when the file stores only LoRA/head trainable weights.
    t = t.to(dev).eval()
    for p in t.parameters():
        p.requires_grad = False
    print(f"teacher: {path} | params {sum(p.numel() for p in t.parameters())/1e6:.1f}M")
    print(f"teacher checkpoint load: missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}")
    return t


class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--method", default="full", choices=["full", "lora", "lorma"], help="full fine-tuning, LoRA, or LoRMA+ (multiplicative, additive inflation)")
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--kd", action="store_true", help="knowledge distillation from a fine-tuned teacher")
    ap.add_argument("--kd_teacher_method", default="lora", choices=["full", "lora", "lorma"])
    ap.add_argument("--kd_t_rank", type=int, default=16); ap.add_argument("--kd_t_alpha", type=float, default=32.0)
    ap.add_argument("--kd_T", type=float, default=2.0, help="softmax temperature")
    ap.add_argument("--kd_alpha", type=float, default=0.5, help="weight of hard-label CE; (1-alpha) goes to the KD term")
    ap.add_argument("--kd_teacher_arch", default="arch4", choices=["arch1", "arch2", "arch3", "arch4"],
                    help="teacher backbone architecture; can differ from the student architecture")
    ap.add_argument("--kd_teacher_ckpt", default="", help="explicit path to the fold-matched fine-tuned teacher checkpoint")
    ap.add_argument("--kd_teacher_dir", default=OUT, help="directory containing <teacher_arch>_<teacher_method>_f<fold>_best.pt")
    ap.add_argument("--student_layers", default="", help="optional comma list of student layer indices to retain; empty keeps all student layers")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    if a.kd and a.fold < 0:
        ap.error("KD requires --fold 0..4 so the student uses the matching fold teacher and validation split.")
    n_keep = len([x for x in a.student_layers.split(",") if x.strip()]) if a.student_layers else 0
    tag = (a.arch + ("" if a.method == "full" else f"_{a.method}") + ("_kd" if a.kd else "")
           + (f"_s{n_keep}" if n_keep else "") + (f"_f{a.fold}" if a.fold >= 0 else ""))
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    if not os.path.isfile(CKPT[a.arch]):
        raise FileNotFoundError(f"Student pretrained checkpoint not found: {CKPT[a.arch]}")
    if a.kd and not (a.kd_teacher_ckpt and os.path.isfile(a.kd_teacher_ckpt)):
        expected_teacher = os.path.join(
            a.kd_teacher_dir,
            f"{a.kd_teacher_arch}{'' if a.kd_teacher_method == 'full' else '_' + a.kd_teacher_method}_f{a.fold}_best.pt"
        )
        if not os.path.isfile(expected_teacher):
            raise FileNotFoundError(
                f"KD teacher checkpoint not found. Checked explicit path={a.kd_teacher_ckpt!r} "
                f"and expected path={expected_teacher!r}. Set --kd_teacher_ckpt to the real teacher file."
            )

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    # Build the full pretrained backbone, then retain only the selected layers for the student.
    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    idx = []
    if a.student_layers:
        idx = [int(x.strip()) for x in a.student_layers.split(",") if x.strip()]
        if len(set(idx)) != len(idx):
            raise ValueError(f"Duplicate student layer indices: {idx}")
        inner = getattr(bb, "m", None)
        if inner is None or not hasattr(inner, "layers"):
            raise AttributeError("Student layer surgery expects backbone.m.layers; adapt this block to your Arch4 wrapper.")
        n_layers = len(inner.layers)
        if any(i < 0 or i >= n_layers for i in idx):
            raise IndexError(f"student_layers={idx}, but backbone has {n_layers} layers.")
        inner.layers = nn.ModuleList([inner.layers[i] for i in idx])
        print(f"student: kept layers {idx} from {n_layers} | student backbone layers={len(inner.layers)}")

    # Teacher is built independently with the FULL backbone; never apply student layer pruning to it.
    teacher = build_teacher(
        a.kd_teacher_arch, a.kd_teacher_method, a.fold, dev, a.kd_t_rank, a.kd_t_alpha,
        teacher_ckpt=(a.kd_teacher_ckpt or None), teacher_dir=a.kd_teacher_dir
    ) if a.kd else None
    if a.method in ("lora", "lorma"):
        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout, kind=a.method)
        print(f"{a.method.upper()}: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")
    model = NER(bb, d).to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")
    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            ce_loss = F.cross_entropy(lg.float().reshape(-1, C), lab.reshape(-1), weight=cw, ignore_index=-100)
            kd_loss = torch.zeros((), device=dev)
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                    tl = teacher(ids, msk).float()
                tok_mask = (lab != -100) & (msk != 0)
                if tok_mask.any():
                    T = a.kd_T
                    # KL(teacher || student), computed only on supervised/non-padding tokens.
                    kd_loss = F.kl_div(
                        F.log_softmax(lg.float()[tok_mask] / T, dim=-1),
                        F.softmax(tl[tok_mask] / T, dim=-1),
                        reduction="batchmean"
                    ) * (T * T)
                    loss = a.kd_alpha * ce_loss + (1.0 - a.kd_alpha) * kd_loss
                else:
                    loss = ce_loss
            else:
                loss = ce_loss
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if teacher is not None:
                # Running CE/KD means are useful for checking loss-scale imbalance.
                pass
        if a.final_full:
            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev), strict=False)
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook
    main()

In [26]:
# SINGLE KAGGLE CELL: Arch1 Student + Arch4 LoRA Teacher KD
# Paste this whole cell into Kaggle. No separate pipeline .py upload is needed.
# Prerequisites: build_arch1/build_arch4 (or their modules) and the teacher checkpoint must exist.

PIPELINE_SOURCE = '"""\nFresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip\n\nUsage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12\nOnly thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).\nBackbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)\n"""\nimport argparse, glob, json, math, os, random, sys, zipfile\nfrom collections import defaultdict\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom tokenizers import Tokenizer\n\n# ------------------------------- paths ---------------------------------\nROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"\nTOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"\nCKPT = {\n    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",\n    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",\n    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",\n    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",\n}\nOUT = "/kaggle/working"\n\n\n# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------\ndef build_backbone(arch, ckpt_path, causal=False):\n    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].\n    Example:\n        from my_models import Arch4          # your class\n        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location=\'cpu\')\n        m.load_state_dict(sd.get(\'model\', sd), strict=False)\n        return Wrapper(m), cfg[\'d_model\']    # Wrapper returns final hidden states, NOT LM logits\n    """\n    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}\n    if arch not in mods:\n        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")\n    import importlib\n    name = "build_" + arch\n    try:\n        fn = getattr(importlib.import_module(mods[arch]), name)\n    except (ImportError, AttributeError):  # notebook: run the backbone cell first so build_<arch> is a global\n        fn = globals()[name]\n    return fn(ckpt_path, causal=causal)\n\n\n# ------------------------------- data ----------------------------------\ntok = Tokenizer.from_file(TOK)\nLABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]\nTAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]\nT2I = {t: i for i, t in enumerate(TAGS)}\nC = len(TAGS)\nPAD_ID = 0  # change if your Phase-1 pad id differs\n\n\ndef encode_doc(text, ents=None):\n    enc = tok.encode(text, add_special_tokens=False)\n    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)\n    tags = None\n    if ents is not None:\n        tags = np.zeros(len(ids), dtype=np.int64)\n        for s, e, lab, *_ in ents:\n            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]\n            for k, i in enumerate(idx):\n                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]\n    return ids, offs, tags\n\n\ndef load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):\n    docs = {}\n    for f in ("fold1", "fold2"):\n        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))\n        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))\n        for d in txt:\n            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))\n    keys = sorted(docs)\n    random.Random(seed).shuffle(keys)\n    if fold >= 0:  # document-level K-fold: every doc is validation exactly once\n        vk = set(keys[fold::nfolds])\n        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}\n    nv = max(1, int(len(keys) * val_frac))\n    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}\n\n\ndef make_windows(n, W, stride):\n    if n <= W:\n        return [(0, n)]\n    w = list(range(0, n - W, stride)) + [n - W]\n    return [(s, s + W) for s in w]\n\n\nclass WinDS(torch.utils.data.Dataset):\n    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):\n        rng = random.Random(seed)\n        self.items = []\n        for ids, offs, tags in enc_docs:\n            for s, e in make_windows(len(ids), W, stride):\n                t = tags[s:e]\n                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows\n                    self.items.append((ids[s:e], t))\n\n    def __len__(self): return len(self.items)\n    def __getitem__(self, i): return self.items[i]\n\n\ndef collate(batch):\n    L = max(len(x[0]) for x in batch)\n    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)\n    lab = torch.full((len(batch), L), -100, dtype=torch.long)\n    msk = torch.zeros((len(batch), L), dtype=torch.long)\n    for i, (a, t) in enumerate(batch):\n        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1\n    return ids, msk, lab\n\n\n# ------------------------------- model ---------------------------------\nclass LoRALinear(nn.Module):\n    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, only A,B trained. B starts at 0 => identical to base at step 0."""\n    def __init__(self, base, r, alpha, dropout):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.empty(r, base.in_features))\n        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x):\n        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale\n\n\nclass LoRMAPlusLinear(nn.Module):\n    """LoRMA+ (Low-Rank Multiplicative Adaptation, additive inflation):  W = (I + (alpha/r) B A) W0.\n    With h = W0 x (frozen layer output):  y = h + (alpha/r) * B(A h).  A:(r,out)  B:(out,r), B=0 at init => identity => y = W0 x."""\n    def __init__(self, base, r, alpha, dropout=0.0):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.empty(r, base.out_features))\n        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n\n    def forward(self, x):\n        h = self.base(x)\n        return h + (h @ self.A.t() @ self.B.t()) * self.scale\n\n\nLORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")\n\n\ndef apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS, kind="lora"):\n    for p in backbone.parameters(): p.requires_grad = False  # freeze everything in the Phase-1 model\n    n = 0\n    for name, mod in list(backbone.named_modules()):\n        for cname, child in list(mod.named_children()):\n            if isinstance(child, nn.Linear) and cname in targets:\n                cls = LoRMAPlusLinear if kind == "lorma" else LoRALinear\n                setattr(mod, cname, cls(child, r, alpha, dropout)); n += 1\n    return n\n\n\ndef ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files\n    keep = {n for n, p in model.named_parameters() if p.requires_grad}\n    return {k: v for k, v in model.state_dict().items() if k in keep}\n\n\ndef build_teacher(arch, method, fold, dev, rank, alpha, teacher_ckpt=None, teacher_dir=OUT):\n    """Load a frozen, fold-matched NER teacher. Explicit --kd_teacher_ckpt overrides directory naming."""\n    bb_t, d_t = build_backbone(arch, CKPT[arch], causal=False)\n    if method in ("lora", "lorma"):\n        apply_lora(bb_t, rank, alpha, 0.0, kind=method)\n    t = NER(bb_t, d_t)\n    path = teacher_ckpt or os.path.join(\n        teacher_dir, f"{arch}{\'\' if method == \'full\' else \'_\' + method}_f{fold}_best.pt"\n    )\n    if not os.path.isfile(path):\n        raise FileNotFoundError(\n            f"KD teacher checkpoint not found: {path}. Pass --kd_teacher_ckpt /path/to/teacher.pt "\n            "or set --kd_teacher_dir to the directory containing fold teacher checkpoints."\n        )\n    state = torch.load(path, map_location="cpu")\n    if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):\n        state = state["model"]\n    res = t.load_state_dict(state, strict=False)\n    if res.unexpected_keys:\n        raise RuntimeError(f"Teacher checkpoint has unexpected keys: {res.unexpected_keys[:10]}")\n    # Missing frozen base weights can be expected when the file stores only LoRA/head trainable weights.\n    t = t.to(dev).eval()\n    for p in t.parameters():\n        p.requires_grad = False\n    print(f"teacher: {path} | params {sum(p.numel() for p in t.parameters())/1e6:.1f}M")\n    print(f"teacher checkpoint load: missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}")\n    return t\n\n\nclass NER(nn.Module):\n    def __init__(self, backbone, d):\n        super().__init__()\n        self.bb = backbone\n        self.drop = nn.Dropout(0.1)\n        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))\n\n    def forward(self, ids, msk):\n        return self.head(self.drop(self.bb(ids, msk)))\n\n\n# --------------------- constrained decoding (whole doc) -----------------\ndef trans_mask():\n    M = np.zeros((C, C), dtype=np.float32)\n    for a, ta in enumerate(TAGS):\n        for b, tb in enumerate(TAGS):\n            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):\n                M[a, b] = -1e4\n    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)\n    return M, start\n\n\nTM, START = trans_mask()\n\n\ndef viterbi(em):  # em [T,C] log-probs\n    T = len(em)\n    if T == 0:\n        return []\n    score = START + em[0]\n    back = np.zeros((T, C), dtype=np.int64)\n    for t in range(1, T):\n        cand = score[:, None] + TM\n        back[t] = cand.argmax(0)\n        score = cand.max(0) + em[t]\n    path = [int(score.argmax())]\n    for t in range(T - 1, 0, -1):\n        path.append(int(back[t][path[-1]]))\n    return path[::-1]\n\n\ndef tags_to_spans(path, offs, text):\n    spans, cur = [], None\n    for (a, b), ti in zip(offs, path):\n        t = TAGS[ti]\n        if t == "O":\n            if cur: spans.append(cur); cur = None\n            continue\n        lab = t[2:]\n        if t.startswith("B-") or cur is None or cur[2] != lab:\n            if cur: spans.append(cur)\n            cur = [int(a), int(b), lab]\n        else:\n            cur[1] = int(b)\n    if cur: spans.append(cur)\n    out = []\n    for s, e, lab in spans:\n        while s < e and text[s].isspace(): s += 1\n        while e > s and text[e - 1].isspace(): e -= 1\n        if e > s: out.append((s, e, lab))\n    return out\n\n\n@torch.no_grad()\ndef predict_doc(model, ids, W, stride, dev, bs=8):\n    model.eval()\n    n = len(ids)\n    if n == 0:  # empty / whitespace-only document\n        return np.zeros((0, C), dtype=np.float32)\n    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)\n    wins = make_windows(n, W, stride)\n    for i in range(0, len(wins), bs):\n        chunk = wins[i:i + bs]\n        L = max(e - s for s, e in chunk)\n        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)\n        for j, (s, e) in enumerate(chunk):\n            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):\n            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()\n        for j, (s, e) in enumerate(chunk):\n            l = e - s\n            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context\n            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w\n    return acc / np.maximum(wsum, 1e-6)[:, None]\n\n\ndef strict_f1(pred, gold):\n    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)\n    for d in gold:\n        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}\n        for x in p & g: tp[x[2]] += 1\n        for x in p - g: fp[x[2]] += 1\n        for x in g - p: fn[x[2]] += 1\n    rows, fs = {}, []\n    for l in LABELS:\n        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)\n        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)\n    return float(np.mean(fs)), rows\n\n\ndef evaluate(model, docs, encs, W, stride, dev):\n    pred = {}\n    for k, (text, _) in docs.items():\n        ids, offs, _t = encs[k]\n        em = predict_doc(model, ids, W, stride, dev)\n        pred[k] = tags_to_spans(viterbi(em), offs, text)\n    return strict_f1(pred, {k: v[1] for k, v in docs.items()})\n\n\n# ------------------------------- main ----------------------------------\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)\n    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)\n    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)\n    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",\n        help="after picking epochs, retrain on ALL train docs (no validation)")\n    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")\n    ap.add_argument("--method", default="full", choices=["full", "lora", "lorma"], help="full fine-tuning, LoRA, or LoRMA+ (multiplicative, additive inflation)")\n    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)\n    ap.add_argument("--lora_dropout", type=float, default=0.05)\n    ap.add_argument("--kd", action="store_true", help="knowledge distillation from a fine-tuned teacher")\n    ap.add_argument("--kd_teacher_method", default="lora", choices=["full", "lora", "lorma"])\n    ap.add_argument("--kd_t_rank", type=int, default=16); ap.add_argument("--kd_t_alpha", type=float, default=32.0)\n    ap.add_argument("--kd_T", type=float, default=2.0, help="softmax temperature")\n    ap.add_argument("--kd_alpha", type=float, default=0.5, help="weight of hard-label CE; (1-alpha) goes to the KD term")\n    ap.add_argument("--kd_teacher_arch", default="arch4", choices=["arch1", "arch2", "arch3", "arch4"],\n                    help="teacher backbone architecture; can differ from the student architecture")\n    ap.add_argument("--kd_teacher_ckpt", default="", help="explicit path to the fold-matched fine-tuned teacher checkpoint")\n    ap.add_argument("--kd_teacher_dir", default=OUT, help="directory containing <teacher_arch>_<teacher_method>_f<fold>_best.pt")\n    ap.add_argument("--student_layers", default="", help="optional comma list of student layer indices to retain; empty keeps all student layers")\n    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)\n    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")\n    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")\n    ap.add_argument("--predict_only", action="store_true",\n                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")\n    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab\'s "-f kernel.json"\n    if a.predict_only: a.epochs = 0\n    if a.kd and a.fold < 0:\n        ap.error("KD requires --fold 0..4 so the student uses the matching fold teacher and validation split.")\n    n_keep = len([x for x in a.student_layers.split(",") if x.strip()]) if a.student_layers else 0\n    tag = (a.arch + ("" if a.method == "full" else f"_{a.method}") + ("_kd" if a.kd else "")\n           + (f"_s{n_keep}" if n_keep else "") + (f"_f{a.fold}" if a.fold >= 0 else ""))\n    if a.ensemble:  # average per-fold emissions, decode once, write zip\n        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}\n        for d_, text in test.items():\n            ids, offs, _ = encode_doc(text)\n            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]\n            if len(ids) == 0 or not fs: pred[d_] = []; continue\n            em = np.mean(fs, 0)\n            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]\n        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))\n        json.dump(pred, open(f"{OUT}/prediction.json", "w"))\n        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:\n            z.write(f"{OUT}/prediction.json", arcname="prediction.json")\n        return\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    torch.manual_seed(0); np.random.seed(0); random.seed(0)\n\n    if not os.path.isfile(CKPT[a.arch]):\n        raise FileNotFoundError(f"Student pretrained checkpoint not found: {CKPT[a.arch]}")\n    if a.kd and not (a.kd_teacher_ckpt and os.path.isfile(a.kd_teacher_ckpt)):\n        expected_teacher = os.path.join(\n            a.kd_teacher_dir,\n            f"{a.kd_teacher_arch}{\'\' if a.kd_teacher_method == \'full\' else \'_\' + a.kd_teacher_method}_f{a.fold}_best.pt"\n        )\n        if not os.path.isfile(expected_teacher):\n            raise FileNotFoundError(\n                f"KD teacher checkpoint not found. Checked explicit path={a.kd_teacher_ckpt!r} "\n                f"and expected path={expected_teacher!r}. Set --kd_teacher_ckpt to the real teacher file."\n            )\n\n    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)\n    if a.final_full: tr = {**tr, **va}\n    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}\n    va_enc = {k: encode_doc(*v) for k, v in va.items()}\n    ds = WinDS(list(tr_enc.values()), a.W, a.stride)\n    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)\n\n    # class weights: sqrt inverse-frequency, capped; O stays low\n    cnt = np.zeros(C)\n    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)\n    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)\n    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):\n        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost\n    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)\n    cw = torch.tensor(w, dtype=torch.float32, device=dev)\n    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")\n\n    # Build the full pretrained backbone, then retain only the selected layers for the student.\n    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)\n    idx = []\n    if a.student_layers:\n        idx = [int(x.strip()) for x in a.student_layers.split(",") if x.strip()]\n        if len(set(idx)) != len(idx):\n            raise ValueError(f"Duplicate student layer indices: {idx}")\n        inner = getattr(bb, "m", None)\n        if inner is None or not hasattr(inner, "layers"):\n            raise AttributeError("Student layer surgery expects backbone.m.layers; adapt this block to your Arch4 wrapper.")\n        n_layers = len(inner.layers)\n        if any(i < 0 or i >= n_layers for i in idx):\n            raise IndexError(f"student_layers={idx}, but backbone has {n_layers} layers.")\n        inner.layers = nn.ModuleList([inner.layers[i] for i in idx])\n        print(f"student: kept layers {idx} from {n_layers} | student backbone layers={len(inner.layers)}")\n\n    # Teacher is built independently with the FULL backbone; never apply student layer pruning to it.\n    teacher = build_teacher(\n        a.kd_teacher_arch, a.kd_teacher_method, a.fold, dev, a.kd_t_rank, a.kd_t_alpha,\n        teacher_ckpt=(a.kd_teacher_ckpt or None), teacher_dir=a.kd_teacher_dir\n    ) if a.kd else None\n    if a.method in ("lora", "lorma"):\n        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout, kind=a.method)\n        print(f"{a.method.upper()}: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")\n    model = NER(bb, d).to(dev)\n    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    n_all = sum(p.numel() for p in model.parameters())\n    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")\n    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},\n                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)\n    total = a.epochs * len(dl); warm = int(0.06 * total)\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else\n                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))\n    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")\n\n    best, bad = -1, 0\n    for ep in range(a.epochs):\n        model.train(); run = 0\n        for ids, msk, lab in dl:\n            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):\n                lg = model(ids, msk)\n            ce_loss = F.cross_entropy(lg.float().reshape(-1, C), lab.reshape(-1), weight=cw, ignore_index=-100)\n            kd_loss = torch.zeros((), device=dev)\n            if teacher is not None:\n                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):\n                    tl = teacher(ids, msk).float()\n                tok_mask = (lab != -100) & (msk != 0)\n                if tok_mask.any():\n                    T = a.kd_T\n                    # KL(teacher || student), computed only on supervised/non-padding tokens.\n                    kd_loss = F.kl_div(\n                        F.log_softmax(lg.float()[tok_mask] / T, dim=-1),\n                        F.softmax(tl[tok_mask] / T, dim=-1),\n                        reduction="batchmean"\n                    ) * (T * T)\n                    loss = a.kd_alpha * ce_loss + (1.0 - a.kd_alpha) * kd_loss\n                else:\n                    loss = ce_loss\n            else:\n                loss = ce_loss\n            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()\n            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt); scaler.update(); sched.step()\n            run += loss.item()\n            if teacher is not None:\n                # Running CE/KD means are useful for checking loss-scale imbalance.\n                pass\n        if a.final_full:\n            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue\n        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)\n        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")\n        if f1 > best:\n            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")\n            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")\n        else:\n            bad += 1\n            if bad >= 4: print("early stop"); break\n\n    # ---------------- predict test & package ----------------\n    model.load_state_dict(torch.load(f"{OUT}/{tag}_{\'final\' if a.final_full else \'best\'}.pt", map_location=dev), strict=False)\n    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}\n    for d_, text in test.items():\n        ids, offs, _ = encode_doc(text)\n        em = predict_doc(model, ids, a.W, a.stride, dev)\n        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]\n        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling\n    os.makedirs(OUT, exist_ok=True)\n    json.dump(pred, open(f"{OUT}/prediction.json", "w"))\n    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:\n        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root\n    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)\n\n\nif __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook\n    main()'

PIPELINE_PATH = "/kaggle/working/kd_arch1_student_pipeline.py"
import os, sys

with open(PIPELINE_PATH, "w", encoding="utf-8") as f:
    f.write(PIPELINE_SOURCE)

assert os.path.isfile(
    "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt"
), "Arch1 pretrained checkpoint not found."

TEACHER_CKPT = "/kaggle/working/arch4_lora_f0_best.pt"
assert os.path.isfile(TEACHER_CKPT), (
    f"Arch4 teacher checkpoint not found: {TEACHER_CKPT}. "
    "Change TEACHER_CKPT to the real checkpoint path."
)

# Load pipeline functions into this notebook's global namespace.
exec(compile(PIPELINE_SOURCE, PIPELINE_PATH, "exec"), globals())

sys.argv = [
    "kd_arch1_student_pipeline.py",
    "--arch", "arch1",
    "--fold", "0",
    "--nfolds", "5",
    "--epochs", "8",
    "--method", "full",
    "--lr_bb", "5e-5",
    "--lr_head", "1e-3",
    "--kd",
    "--kd_teacher_arch", "arch4",
    "--kd_teacher_method", "lora",
    "--kd_teacher_ckpt", TEACHER_CKPT,
    "--kd_t_rank", "16",
    "--kd_t_alpha", "32",
    "--kd_T", "2.0",
    "--kd_alpha", "0.5",
    "--role_boost", "2.0",
    "--W", "512",
    "--stride", "384",
    "--bs", "8"
]
main()


WIT training tokens: 476 | role_boost 2.0
train docs 56 val docs 14 windows 891


KeyError: 'build_arch1'

In [27]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"}]
    for file in files:
        if file.endswith(".py") and (
            "arch1" in file.lower() or "model" in file.lower()
        ):
            print(os.path.join(root, file))

/kaggle/working/kd_arch1_student_pipeline.py


In [28]:
print("build_arch1:", "build_arch1" in globals())

print(
    "Arch1-related objects:",
    [name for name in globals() if "arch1" in name.lower()]
)

build_arch1: False
Arch1-related objects: []


In [29]:
# Run this ONE diagnostic cell in Kaggle.
# It searches likely mounted model files and prints checkpoint key/shape samples.
import os, glob, torch

ARCH1_CKPT = "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt"

print("Checkpoint exists:", os.path.isfile(ARCH1_CKPT))
print("\nPython files in /kaggle/input with arch1/model/decoder names:")
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for f in files:
        low = f.lower()
        if f.endswith(".py") and any(x in low for x in ("arch1", "model", "decoder")):
            print(os.path.join(root, f))

print("\nCheckpoint structure:")
obj = torch.load(ARCH1_CKPT, map_location="cpu", weights_only=False)
if isinstance(obj, dict):
    print("Top-level keys:", list(obj.keys())[:30])
    sd = obj.get("model", obj.get("state_dict", obj))
    if isinstance(sd, dict):
        print("State-dict tensor keys:", sum(torch.is_tensor(v) for v in sd.values()))
        for k, v in list(sd.items())[:80]:
            if torch.is_tensor(v):
                print(f"{k:65s} {tuple(v.shape)}")
else:
    print("Object type:", type(obj))

print("\nPotential model source files in /kaggle/working:")
for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"}]
    for f in files:
        if f.endswith((".py", ".ipynb")) and f != "kd_arch1_student_pipeline.py":
            print(os.path.join(root, f))


Checkpoint exists: True

Python files in /kaggle/input with arch1/model/decoder names:

Checkpoint structure:
Top-level keys: ['model', 'opt', 'scaler', 'step', 'best', 'cfg', 'n_params']
State-dict tensor keys: 141
token_emb.weight                                                  (40000, 768)
pos_emb.weight                                                    (1024, 768)
blocks.0.norm1.weight                                             (768,)
blocks.0.norm1.bias                                               (768,)
blocks.0.attn.mask                                                (1, 1, 1024, 1024)
blocks.0.attn.q_proj.weight                                       (768, 768)
blocks.0.attn.q_proj.bias                                         (768,)
blocks.0.attn.k_proj.weight                                       (768, 768)
blocks.0.attn.k_proj.bias                                         (768,)
blocks.0.attn.v_proj.weight                                       (768, 768)
blocks.0.attn.v_pro

In [30]:
import torch

ckpt_path = (
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("========== ARCH 1 CONFIG ==========")
print(ckpt["cfg"])

print("\n========== CHECKPOINT METADATA ==========")
print("Step:", ckpt.get("step"))
print("Best:", ckpt.get("best"))
print("Parameter count:", ckpt.get("n_params"))

print("\n========== REMAINING MODEL KEYS ==========")
sd = ckpt["model"]

for name, tensor in sd.items():
    if name.startswith(("blocks.5.", "blocks.6.", "blocks.7.")) or name.startswith(
        ("final_norm.", "norm.", "lm_head.", "token_emb.", "pos_emb.")
    ):
        print(name, tuple(tensor.shape))

print("\n========== LAYER COUNT ==========")
layers = {
    int(name.split(".")[1])
    for name in sd
    if name.startswith("blocks.") and name.split(".")[1].isdigit()
}
print("Transformer layers:", sorted(layers))

========== ARCH 1 CONFIG ==========
{'BLOCK': 1024, 'MICRO_BATCH': 1, 'GRAD_ACCUM': 16, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}

========== CHECKPOINT METADATA ==========
Step: 10000
Best: 4.100625859101613
Parameter count: 88210944

========== REMAINING MODEL KEYS ==========
token_emb.weight (40000, 768)
pos_emb.weight (1024, 768)
blocks.5.norm1.weight (768,)
blocks.5.norm1.bias (768,)
blocks.5.attn.mask (1, 1, 1024, 1024)
blocks.5.attn.q_proj.weight (768, 768)
blocks.5.attn.q_proj.bias (768,)
blocks.5.attn.k_proj.weight (768, 768)
blocks.5.attn.k_proj.bias (768,)
blocks.5.attn.v_proj.weight (768, 768)
blocks.5.attn.v_proj.bias (768,)
blocks.5.attn.out_proj.weight (768, 768)
blocks.5.attn.out_proj.bias (768,)
blocks.5.norm2.weight (768,)
blocks.5.norm2.bias (768,)
blocks.5.mlp.fc1.weight (3072, 768)
blocks.5.mlp.fc1.bias (3072,)
blocks.5.mlp.fc2.weight (76

In [34]:
"""
Fresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip

Usage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12
Only thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).
Backbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)
"""
import argparse, glob, json, math, os, random, sys, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

# ------------------------------- paths ---------------------------------
ROOT = "/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"
TOK = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT = {
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
OUT = "/kaggle/working"


# ------------------- >>> FILL THIS IN (your Phase-1 model) <<< -----------
def build_backbone(arch, ckpt_path, causal=False):
    """Return (module, hidden_dim). Must map (input_ids, attention_mask) -> [B,T,D].
    Example:
        from my_models import Arch4          # your class
        m = Arch4(**cfg); sd = torch.load(ckpt_path, map_location='cpu')
        m.load_state_dict(sd.get('model', sd), strict=False)
        return Wrapper(m), cfg['d_model']    # Wrapper returns final hidden states, NOT LM logits
    """
    mods = {"arch4": "arch4_backbone", "arch1": "arch1_backbone", "arch3": "arch3_backbone"}
    if arch not in mods:
        raise NotImplementedError(f"{arch}: send me its checkpoint shapes/cfg first")
    import importlib
    name = "build_" + arch
    try:
        fn = getattr(importlib.import_module(mods[arch]), name)
    except (ImportError, AttributeError):  # notebook: run the backbone cell first so build_<arch> is a global
        fn = globals()[name]
    return fn(ckpt_path, causal=causal)


# ------------------------------- data ----------------------------------
tok = Tokenizer.from_file(TOK)
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
TAGS = ["O"] + [p + l for l in LABELS for p in ("B-", "I-")]
T2I = {t: i for i, t in enumerate(TAGS)}
C = len(TAGS)
PAD_ID = 0  # change if your Phase-1 pad id differs


def encode_doc(text, ents=None):
    enc = tok.encode(text, add_special_tokens=False)
    ids, offs = np.array(enc.ids, dtype=np.int64), np.array(enc.offsets, dtype=np.int64)
    tags = None
    if ents is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        for s, e, lab, *_ in ents:
            idx = np.where((offs[:, 0] < e) & (offs[:, 1] > s))[0]
            for k, i in enumerate(idx):
                tags[i] = T2I[("B-" if k == 0 else "I-") + lab]
    return ids, offs, tags


def load_split(seed=13, val_frac=0.12, fold=-1, nfolds=5):
    docs = {}
    for f in ("fold1", "fold2"):
        txt = json.load(open(f"{ROOT}/Train/{f}-text.json"))
        gold = json.load(open(f"{ROOT}/Train/{f}-gold.json"))
        for d in txt:
            docs[f"{f}_{d}"] = (txt[d], gold.get(d, []))
    keys = sorted(docs)
    random.Random(seed).shuffle(keys)
    if fold >= 0:  # document-level K-fold: every doc is validation exactly once
        vk = set(keys[fold::nfolds])
        return {k: docs[k] for k in keys if k not in vk}, {k: docs[k] for k in keys if k in vk}
    nv = max(1, int(len(keys) * val_frac))
    return {k: docs[k] for k in keys[nv:]}, {k: docs[k] for k in keys[:nv]}


def make_windows(n, W, stride):
    if n <= W:
        return [(0, n)]
    w = list(range(0, n - W, stride)) + [n - W]
    return [(s, s + W) for s in w]


class WinDS(torch.utils.data.Dataset):
    def __init__(self, enc_docs, W, stride, neg_keep=0.3, seed=0):
        rng = random.Random(seed)
        self.items = []
        for ids, offs, tags in enc_docs:
            for s, e in make_windows(len(ids), W, stride):
                t = tags[s:e]
                if (t > 0).any() or rng.random() < neg_keep:  # downsample empty windows
                    self.items.append((ids[s:e], t))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def collate(batch):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    msk = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (a, t) in enumerate(batch):
        ids[i, :len(a)] = torch.from_numpy(a); lab[i, :len(a)] = torch.from_numpy(t); msk[i, :len(a)] = 1
    return ids, msk, lab


# ------------------------------- model ---------------------------------
def _linear_dims(base):
    """Return (in_features, out_features) for nn.Linear and compatible custom projections."""
    in_f = getattr(base, "in_features", None)
    out_f = getattr(base, "out_features", None)
    weight = getattr(base, "weight", None)
    if (in_f is None or out_f is None) and isinstance(weight, torch.Tensor) and weight.ndim == 2:
        out_f, in_f = weight.shape
    if in_f is None or out_f is None or not callable(base):
        raise TypeError(f"LoRA target {type(base).__name__} has no usable 2-D linear weight/dimensions")
    return int(in_f), int(out_f)


class LoRALinear(nn.Module):
    """LoRA wrapper compatible with nn.Linear and custom linear-like projection modules."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.in_features, self.out_features = _linear_dims(base)
        self.A = nn.Parameter(torch.empty(r, self.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = (I + (alpha/r) B A) W0, implemented on the frozen projection output."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        _in_f, self.out_features = _linear_dims(base)
        self.A = nn.Parameter(torch.empty(r, self.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + (h @ self.A.t() @ self.B.t()) * self.scale


LORA_TARGETS = ("qkv_proj", "q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "fc1", "fc2")


def apply_lora(backbone, r, alpha, dropout, targets=LORA_TARGETS, kind="lora"):
    """Inject adapters into named linear-like projections, including compatible custom Linear classes."""
    for p in backbone.parameters():
        p.requires_grad = False
    n = 0
    for _name, mod in list(backbone.named_modules()):
        for cname, child in list(mod.named_children()):
            if cname not in targets or isinstance(child, (LoRALinear, LoRMAPlusLinear)):
                continue
            weight = getattr(child, "weight", None)
            is_linear_like = isinstance(child, nn.Linear) or (
                isinstance(weight, torch.Tensor) and weight.ndim == 2 and callable(child)
            )
            if is_linear_like:
                cls = LoRMAPlusLinear if kind == "lorma" else LoRALinear
                setattr(mod, cname, cls(child, r, alpha, dropout))
                n += 1
    if n == 0:
        names = [name for name, _ in backbone.named_modules() if name.rsplit(".", 1)[-1] in targets]
        raise RuntimeError(
            "LoRA injection wrapped 0 layers. The current backbone projection names/types do not match "
            f"the adapter checkpoint. Target-like module names: {names[:20]}"
        )
    return n


def ckpt_state(model):  # save only what trains (LoRA: A,B + head) -> tiny files
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


def build_teacher(arch, method, fold, dev, rank, alpha, teacher_ckpt=None, teacher_dir=OUT):
    """Load a frozen, fold-matched NER teacher. Explicit --kd_teacher_ckpt overrides directory naming."""
    bb_t, d_t = build_backbone(arch, CKPT[arch], causal=False)
    if method in ("lora", "lorma"):
        n_wrapped = apply_lora(bb_t, rank, alpha, 0.0, kind=method)
        print(f"Teacher {method.upper()} adapters injected: {n_wrapped} projection layers | rank={rank} alpha={alpha}")
    t = NER(bb_t, d_t)
    path = teacher_ckpt or os.path.join(
        teacher_dir, f"{arch}{'' if method == 'full' else '_' + method}_f{fold}_best.pt"
    )
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Pass --kd_teacher_ckpt /path/to/teacher.pt "
            "or set --kd_teacher_dir to the directory containing fold teacher checkpoints."
        )
    state = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(state, dict):
        # Support common checkpoint containers without silently loading the wrong object.
        for container_key in ("model", "state_dict", "model_state_dict"):
            if container_key in state and isinstance(state[container_key], dict):
                state = state[container_key]
                break
    if not isinstance(state, dict):
        raise TypeError(f"Teacher checkpoint must contain a state_dict; got {type(state).__name__}: {path}")
    # Normalize DataParallel prefixes if present.
    if state and all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}
    res = t.load_state_dict(state, strict=False)
    if res.unexpected_keys:
        adapter_unexpected = [k for k in res.unexpected_keys if k.endswith(".A") or k.endswith(".B")]
        raise RuntimeError(
            f"Teacher checkpoint has {len(res.unexpected_keys)} unexpected keys. "
            f"Unexpected LoRA keys={adapter_unexpected[:8]}. "
            "This usually means the teacher's adapter modules were not injected with the same projection names, "
            "or --kd_teacher_method/rank does not match the saved teacher. "
            f"First unexpected keys: {res.unexpected_keys[:12]}"
        )
    if method in ("lora", "lorma"):
        expected_adapter = [k for k, _ in t.named_parameters() if k.endswith(".A") or k.endswith(".B")]
        saved_adapter = [k for k in state if k.endswith(".A") or k.endswith(".B")]
        if not expected_adapter:
            raise RuntimeError("Teacher model has no adapter parameters after injection; refusing to run KD with an unadapted teacher.")
        if saved_adapter:
            loaded_adapters = [k for k in expected_adapter if k in state]
            if not loaded_adapters:
                raise RuntimeError(
                    "No teacher adapter keys matched the checkpoint. Check teacher checkpoint architecture and "
                    "the --kd_teacher_method / rank settings. "
                    f"Expected examples={expected_adapter[:5]}, saved examples={saved_adapter[:5]}"
                )
        print(f"Teacher adapter checkpoint keys: saved={len(saved_adapter)}, matched={sum(k in state for k in expected_adapter)}")
    # Missing frozen base weights are expected for adapter-only checkpoints, because the base was loaded from Phase 1.
    t = t.to(dev).eval()
    for p in t.parameters():
        p.requires_grad = False
    print(f"teacher: {path} | params {sum(p.numel() for p in t.parameters())/1e6:.1f}M")
    print(f"teacher checkpoint load: missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}")
    return t


class NER(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, C))

    def forward(self, ids, msk):
        return self.head(self.drop(self.bb(ids, msk)))


# --------------------- constrained decoding (whole doc) -----------------
def trans_mask():
    M = np.zeros((C, C), dtype=np.float32)
    for a, ta in enumerate(TAGS):
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-") and not (ta != "O" and ta[2:] == tb[2:]):
                M[a, b] = -1e4
    start = np.array([-1e4 if t.startswith("I-") else 0 for t in TAGS], dtype=np.float32)
    return M, start


TM, START = trans_mask()


def viterbi(em):  # em [T,C] log-probs
    T = len(em)
    if T == 0:
        return []
    score = START + em[0]
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TM
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return path[::-1]


def tags_to_spans(path, offs, text):
    spans, cur = [], None
    for (a, b), ti in zip(offs, path):
        t = TAGS[ti]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        lab = t[2:]
        if t.startswith("B-") or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [int(a), int(b), lab]
        else:
            cur[1] = int(b)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out


@torch.no_grad()
def predict_doc(model, ids, W, stride, dev, bs=8):
    model.eval()
    n = len(ids)
    if n == 0:  # empty / whitespace-only document
        return np.zeros((0, C), dtype=np.float32)
    acc = np.zeros((n, C), dtype=np.float32); wsum = np.zeros(n, dtype=np.float32)
    wins = make_windows(n, W, stride)
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        L = max(e - s for s, e in chunk)
        x = torch.full((len(chunk), L), PAD_ID, dtype=torch.long); m = torch.zeros_like(x)
        for j, (s, e) in enumerate(chunk):
            x[j, :e - s] = torch.from_numpy(ids[s:e]); m[j, :e - s] = 1
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            lp = F.log_softmax(model(x.to(dev), m.to(dev)).float(), -1).cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            l = e - s
            w = 1.0 - 0.9 * np.abs(np.linspace(-1, 1, l))  # centre-weighted: edges have less context
            acc[s:e] += lp[j, :l] * w[:, None]; wsum[s:e] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def strict_f1(pred, gold):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in gold:
        p = set(pred.get(d, [])); g = {(e[0], e[1], e[2]) for e in gold[d]}
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows, fs = {}, []
    for l in LABELS:
        pr = tp[l] / max(tp[l] + fp[l], 1); rc = tp[l] / max(tp[l] + fn[l], 1)
        f = 2 * pr * rc / max(pr + rc, 1e-9); fs.append(f); rows[l] = (pr, rc, f)
    return float(np.mean(fs)), rows


def evaluate(model, docs, encs, W, stride, dev):
    pred = {}
    for k, (text, _) in docs.items():
        ids, offs, _t = encs[k]
        em = predict_doc(model, ids, W, stride, dev)
        pred[k] = tags_to_spans(viterbi(em), offs, text)
    return strict_f1(pred, {k: v[1] for k, v in docs.items()})


# ------------------------------- main ----------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="arch4"); ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--W", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--bs", type=int, default=8); ap.add_argument("--lr_bb", type=float, default=5e-5)
    ap.add_argument("--lr_head", type=float, default=1e-3); ap.add_argument("--final_full", action="store_true",
        help="after picking epochs, retrain on ALL train docs (no validation)")
    ap.add_argument("--causal", action="store_true", help="keep the backbone causal (as pretrained); use for decoder models")
    ap.add_argument("--method", default="full", choices=["full", "lora", "lorma"], help="full fine-tuning, LoRA, or LoRMA+ (multiplicative, additive inflation)")
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--kd", action="store_true", help="knowledge distillation from a fine-tuned teacher")
    ap.add_argument("--kd_teacher_method", default="lora", choices=["full", "lora", "lorma"])
    ap.add_argument("--kd_t_rank", type=int, default=16); ap.add_argument("--kd_t_alpha", type=float, default=32.0)
    ap.add_argument("--kd_T", type=float, default=2.0, help="softmax temperature")
    ap.add_argument("--kd_alpha", type=float, default=0.5, help="weight of hard-label CE; (1-alpha) goes to the KD term")
    ap.add_argument("--kd_teacher_arch", default="arch4", choices=["arch1", "arch2", "arch3", "arch4"],
                    help="teacher backbone architecture; can differ from the student architecture")
    ap.add_argument("--kd_teacher_ckpt", default="", help="explicit path to the fold-matched fine-tuned teacher checkpoint")
    ap.add_argument("--kd_teacher_dir", default=OUT, help="directory containing <teacher_arch>_<teacher_method>_f<fold>_best.pt")
    ap.add_argument("--student_layers", default="", help="optional comma list of student layer indices to retain; empty keeps all student layers")
    ap.add_argument("--fold", type=int, default=-1); ap.add_argument("--nfolds", type=int, default=5)
    ap.add_argument("--role_boost", type=float, default=1.0, help="extra loss weight for APP/RESP/*.COUNSEL/WIT")
    ap.add_argument("--ensemble", action="store_true", help="average saved fold emissions and write submission")
    ap.add_argument("--predict_only", action="store_true",
                    help="skip training; load <arch>_best.pt (or _final.pt with --final_full) and write the submission")
    a, _unknown = ap.parse_known_args()  # ignores Jupyter/Colab's "-f kernel.json"
    if a.predict_only: a.epochs = 0
    if a.kd and a.fold < 0:
        ap.error("KD requires --fold 0..4 so the student uses the matching fold teacher and validation split.")
    n_keep = len([x for x in a.student_layers.split(",") if x.strip()]) if a.student_layers else 0
    tag = (a.arch + ("" if a.method == "full" else f"_{a.method}") + ("_kd" if a.kd else "")
           + (f"_s{n_keep}" if n_keep else "") + (f"_f{a.fold}" if a.fold >= 0 else ""))
    if a.ensemble:  # average per-fold emissions, decode once, write zip
        test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
        for d_, text in test.items():
            ids, offs, _ = encode_doc(text)
            fs = [np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{OUT}/em_{tag}_f*_{d_}.npy"))]
            if len(ids) == 0 or not fs: pred[d_] = []; continue
            em = np.mean(fs, 0)
            pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        print("models averaged:", len(fs), "| docs:", len(pred), "| entities:", sum(map(len, pred.values())))
        json.dump(pred, open(f"{OUT}/prediction.json", "w"))
        with zipfile.ZipFile(f"{OUT}/submission_{tag}_ens.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f"{OUT}/prediction.json", arcname="prediction.json")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0); np.random.seed(0); random.seed(0)

    if not os.path.isfile(CKPT[a.arch]):
        raise FileNotFoundError(f"Student pretrained checkpoint not found: {CKPT[a.arch]}")
    if a.kd and not (a.kd_teacher_ckpt and os.path.isfile(a.kd_teacher_ckpt)):
        expected_teacher = os.path.join(
            a.kd_teacher_dir,
            f"{a.kd_teacher_arch}{'' if a.kd_teacher_method == 'full' else '_' + a.kd_teacher_method}_f{a.fold}_best.pt"
        )
        if not os.path.isfile(expected_teacher):
            raise FileNotFoundError(
                f"KD teacher checkpoint not found. Checked explicit path={a.kd_teacher_ckpt!r} "
                f"and expected path={expected_teacher!r}. Set --kd_teacher_ckpt to the real teacher file."
            )

    tr, va = load_split(fold=a.fold, nfolds=a.nfolds)
    if a.final_full: tr = {**tr, **va}
    tr_enc = {k: encode_doc(*v) for k, v in tr.items()}
    va_enc = {k: encode_doc(*v) for k, v in va.items()}
    ds = WinDS(list(tr_enc.values()), a.W, a.stride)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.bs, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

    # class weights: sqrt inverse-frequency, capped; O stays low
    cnt = np.zeros(C)
    for ids, offs, t in tr_enc.values(): cnt += np.bincount(t, minlength=C)
    w = np.sqrt(cnt.sum() / np.maximum(cnt, 1)); w = w / w[1:].mean(); w[0] = 0.3; w = np.clip(w, 0.3, 6.0)
    for l in ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"):
        w[T2I["B-" + l]] *= a.role_boost; w[T2I["I-" + l]] *= a.role_boost
    print("WIT training tokens:", int(cnt[T2I["B-WIT"]] + cnt[T2I["I-WIT"]]), "| role_boost", a.role_boost)
    cw = torch.tensor(w, dtype=torch.float32, device=dev)
    print(f"train docs {len(tr)} val docs {len(va)} windows {len(ds)}")

    # Build the full pretrained backbone, then retain only the selected layers for the student.
    bb, d = build_backbone(a.arch, CKPT[a.arch], causal=a.causal)
    idx = []
    if a.student_layers:
        idx = [int(x.strip()) for x in a.student_layers.split(",") if x.strip()]
        if len(set(idx)) != len(idx):
            raise ValueError(f"Duplicate student layer indices: {idx}")
        inner = getattr(bb, "m", None)
        if inner is None or not hasattr(inner, "layers"):
            raise AttributeError("Student layer surgery expects backbone.m.layers; adapt this block to your Arch4 wrapper.")
        n_layers = len(inner.layers)
        if any(i < 0 or i >= n_layers for i in idx):
            raise IndexError(f"student_layers={idx}, but backbone has {n_layers} layers.")
        inner.layers = nn.ModuleList([inner.layers[i] for i in idx])
        print(f"student: kept layers {idx} from {n_layers} | student backbone layers={len(inner.layers)}")

    # Teacher is built independently with the FULL backbone; never apply student layer pruning to it.
    teacher = build_teacher(
        a.kd_teacher_arch, a.kd_teacher_method, a.fold, dev, a.kd_t_rank, a.kd_t_alpha,
        teacher_ckpt=(a.kd_teacher_ckpt or None), teacher_dir=a.kd_teacher_dir
    ) if a.kd else None
    if a.method in ("lora", "lorma"):
        n_wrapped = apply_lora(bb, a.rank, a.alpha, a.lora_dropout, kind=a.method)
        print(f"{a.method.upper()}: wrapped {n_wrapped} linear layers | rank {a.rank} alpha {a.alpha}")
    model = NER(bb, d).to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M total ({100*n_tr/n_all:.2f}%)")
    opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": a.lr_bb},
                             {"params": list(model.head.parameters()), "lr": a.lr_head}], weight_decay=0.01)
    total = a.epochs * len(dl); warm = int(0.06 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")

    best, bad = -1, 0
    for ep in range(a.epochs):
        model.train(); run = 0
        for ids, msk, lab in dl:
            ids, msk, lab = ids.to(dev), msk.to(dev), lab.to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lg = model(ids, msk)
            ce_loss = F.cross_entropy(lg.float().reshape(-1, C), lab.reshape(-1), weight=cw, ignore_index=-100)
            kd_loss = torch.zeros((), device=dev)
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                    tl = teacher(ids, msk).float()
                tok_mask = (lab != -100) & (msk != 0)
                if tok_mask.any():
                    T = a.kd_T
                    # KL(teacher || student), computed only on supervised/non-padding tokens.
                    kd_loss = F.kl_div(
                        F.log_softmax(lg.float()[tok_mask] / T, dim=-1),
                        F.softmax(tl[tok_mask] / T, dim=-1),
                        reduction="batchmean"
                    ) * (T * T)
                    loss = a.kd_alpha * ce_loss + (1.0 - a.kd_alpha) * kd_loss
                else:
                    loss = ce_loss
            else:
                loss = ce_loss
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if teacher is not None:
                # Running CE/KD means are useful for checking loss-scale imbalance.
                pass
        if a.final_full:
            torch.save(ckpt_state(model), f"{OUT}/{tag}_final.pt"); print(f"ep{ep} loss {run/len(dl):.4f}"); continue
        f1, rows = evaluate(model, va, va_enc, a.W, a.stride, dev)
        print(f"ep{ep} loss {run/len(dl):.4f} val strict macroF1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0; torch.save(ckpt_state(model), f"{OUT}/{tag}_best.pt")
            for l, (p, r, f) in rows.items(): print(f"   {l:10s} P={p:.3f} R={r:.3f} F1={f:.3f}")
        else:
            bad += 1
            if bad >= 4: print("early stop"); break

    # ---------------- predict test & package ----------------
    model.load_state_dict(torch.load(f"{OUT}/{tag}_{'final' if a.final_full else 'best'}.pt", map_location=dev), strict=False)
    test = json.load(open(f"{ROOT}/Test/fold3-text.json")); pred = {}
    for d_, text in test.items():
        ids, offs, _ = encode_doc(text)
        em = predict_doc(model, ids, a.W, a.stride, dev)
        pred[d_] = [[s, e, l, text[s:e]] for s, e, l in tags_to_spans(viterbi(em), offs, text)]
        np.save(f"{OUT}/em_{tag}_{d_}.npy", em.astype(np.float16))  # for later ensembling
    os.makedirs(OUT, exist_ok=True)
    json.dump(pred, open(f"{OUT}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")  # at zip root
    print("wrote", f"{OUT}/submission_{tag}.zip", "| best val:", best)


if __name__ == "__main__" and "ipykernel" not in sys.modules:  # do not auto-run inside a notebook
    main()

In [36]:
!python /kaggle/working/kd_arch1_student_pipeline.py \
    --arch arch1 \
    --kd \
    --kd_teacher_arch arch4 \
    --kd_teacher_method lora \
    --kd_teacher_ckpt /kaggle/working/arch4_lora_f0_best.pt \
    --kd_t_rank 16 \
    --kd_t_alpha 32 \
    --fold 0 \
    --epochs 8 \
    --W 512 \
    --stride 384 \
    --bs 8

WIT training tokens: 476 | role_boost 1.0
train docs 56 val docs 14 windows 891
Arch1 loaded strictly | step=10000 | params=88,210,944 | tied embeddings
Arch4 loaded strictly | step=10000 | params=201,443,328
Traceback (most recent call last):
  File "/kaggle/working/kd_arch1_student_pipeline.py", line 699, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/kd_arch1_student_pipeline.py", line 621, in main
    teacher = build_teacher(
              ~~~~~~~~~~~~~^
        a.kd_teacher_arch, a.kd_teacher_method, a.fold, dev, a.kd_t_rank, a.kd_t_alpha,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        teacher_ckpt=(a.kd_teacher_ckpt or None), teacher_dir=a.kd_teacher_dir
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ) if a.kd else None
    ^
  File "/kaggle/working/kd_arch1_student_pipeline.py", line 395, in build_teacher
    raise RuntimeError(f"Teacher checkpoint has unexpected keys: {res.unexpected

In [37]:
import sys
import runpy

sys.argv = [
    "kd_arch1_student_pipeline.py",
    "--arch", "arch1",
    "--kd",
    "--kd_teacher_arch", "arch4",
    "--kd_teacher_method", "lora",
    "--kd_teacher_ckpt", "/kaggle/working/arch4_lora_f0_best.pt",
    "--kd_t_rank", "16",
    "--kd_t_alpha", "32",
    "--fold", "0",
    "--epochs", "8",
    "--W", "512",
    "--stride", "384",
    "--bs", "8",
]

runpy.run_path(
    "/kaggle/working/kd_arch1_student_pipeline.py",
    run_name="__main__"
)

{'__name__': '__main__',
 '__doc__': '\nFresh L-NER pipeline (Phase 2). One file: train -> validate (strict F1) -> predict -> submission.zip\n\nUsage (Kaggle):  python lner_pipeline.py --arch arch4 --epochs 12\nOnly thing you must provide: build_backbone() below (your Phase-1 model class + checkpoint loading).\nBackbone contract: forward(input_ids[B,T], attention_mask[B,T]) -> hidden[B,T,D]  (bidirectional preferred)\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/kaggle/working/kd_arch1_student_pipeline.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in valu